# Make chart 1

Overall job count by year

Annual median wage by year

Top occupation groups by employment

Top detailed occupations by employment

Employment trend for top occupation groups

Annual median wage trend for top occupation groups

Top detailed occupations by annual median wage

Top states by employment

Top industries by employment

Top areas by employment

Hourly median wage by year

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import re
import time
import gc

# ============================================================
# JOB CHART + TABLE GENERATOR
# MEMORY OPTIMIZED VERSION
#
# INPUT:
#   Downloads/all_2010-2025_job_CLEANED_numeric_only_1.csv
#
# OUTPUT:
#   Downloads/Job_Chart_1
#   Downloads/Job_Table_1
#
# METHOD:
#   - Reads CSV in chunks
#   - Does NOT load whole file into memory
#   - Creates ranked chart files and ranked table files
#   - Rank 1 = most important
# ============================================================

# ============================================================
# SETTINGS
# ============================================================

CFG = {
    "INPUT_FILE": Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_1.csv",

    "CHART_DIR": Path.home() / "Downloads" / "Job_Chart_1",
    "TABLE_DIR": Path.home() / "Downloads" / "Job_Table_1",

    "CHUNKSIZE": 100_000,

    # Change END_YEAR to 2024 if you do not want 2025
    "START_YEAR": 2010,
    "END_YEAR": 2025,

    "TOP_N": 12,

    # For top wage chart, avoid tiny jobs making weird wage ranking
    "MIN_EMP_FOR_WAGE_TOP": 1_000,

    # From your previous scan
    "EXPECTED_ROWS": 6_597_966,

    "DPI": 180,
    "PRINT_EVERY_CHUNK": 1,
}

CFG["CHART_DIR"].mkdir(parents=True, exist_ok=True)
CFG["TABLE_DIR"].mkdir(parents=True, exist_ok=True)

FILE = CFG["INPUT_FILE"]

if not FILE.exists():
    raise FileNotFoundError(f"File not found:\n{FILE}")

print("=" * 100)
print("JOB CHART + TABLE GENERATOR")
print("=" * 100)
print(f"Input file : {FILE}")
print(f"Chart dir  : {CFG['CHART_DIR']}")
print(f"Table dir  : {CFG['TABLE_DIR']}")
print(f"Chunksize  : {CFG['CHUNKSIZE']:,}")
print(f"Year range : {CFG['START_YEAR']} - {CFG['END_YEAR']}")
print("=" * 100)


# ============================================================
# COLUMNS USED
# Only read needed columns for memory optimization
# ============================================================

USECOLS = [
    "year",
    "area_title",
    "area_type",
    "prim_state",
    "state",
    "state_abbr",
    "naics",
    "naics_title",
    "occ_code",
    "occ_title",
    "group",
    "occ_group",
    "tot_emp",
    "a_median",
    "a_mean",
    "h_median",
    "h_mean",
    "data_level",
]

STRING_COLS = [
    "area_title",
    "prim_state",
    "state",
    "state_abbr",
    "naics",
    "naics_title",
    "occ_code",
    "occ_title",
    "group",
    "occ_group",
    "data_level",
]

DTYPE_MAP = {c: "string" for c in STRING_COLS}


# ============================================================
# HELPERS
# ============================================================

def fmt_number(x, pos=None):
    try:
        return f"{x:,.0f}"
    except Exception:
        return str(x)


def safe_name(title):
    name = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")
    name = re.sub(r"_+", "_", name)
    return name[:140]


def clean_text(s):
    return s.fillna("").astype(str).str.strip()


def clean_lower(s):
    return clean_text(s).str.lower()


def add_sum(parts, name, df, mask, group_cols, value_col, source_type):
    if mask is None or not mask.any():
        return

    cols = group_cols + [value_col]
    sub = df.loc[mask, cols].copy()
    sub = sub.dropna(subset=[value_col])

    if sub.empty:
        return

    if "year" in group_cols:
        sub = sub.dropna(subset=["year"])

    for c in group_cols:
        if c != "year":
            sub[c] = clean_text(sub[c])
            sub = sub[sub[c] != ""]

    if sub.empty:
        return

    g = (
        sub.groupby(group_cols, dropna=False, as_index=False)[value_col]
        .sum()
    )

    g.insert(0, "source_type", source_type)
    parts[name].append(g)


def add_weighted(parts, name, df, mask, group_cols, value_col, weight_col, source_type):
    if mask is None or not mask.any():
        return

    cols = group_cols + [value_col, weight_col]
    sub = df.loc[mask, cols].copy()
    sub = sub.dropna(subset=[value_col, weight_col])
    sub = sub[sub[weight_col] > 0]

    if sub.empty:
        return

    if "year" in group_cols:
        sub = sub.dropna(subset=["year"])

    for c in group_cols:
        if c != "year":
            sub[c] = clean_text(sub[c])
            sub = sub[sub[c] != ""]

    if sub.empty:
        return

    sub["_weighted_sum"] = sub[value_col] * sub[weight_col]

    g = (
        sub.groupby(group_cols, dropna=False, as_index=False)
        .agg(
            weighted_sum=("_weighted_sum", "sum"),
            weight_sum=(weight_col, "sum")
        )
    )

    g.insert(0, "source_type", source_type)
    parts[name].append(g)


def finalize_sum(parts, name, value_col):
    if name not in parts or len(parts[name]) == 0:
        return pd.DataFrame()

    df = pd.concat(parts[name], ignore_index=True)

    key_cols = [c for c in df.columns if c != value_col]

    out = (
        df.groupby(key_cols, dropna=False, as_index=False)[value_col]
        .sum()
    )

    return out


def finalize_weighted(parts, name, output_col):
    if name not in parts or len(parts[name]) == 0:
        return pd.DataFrame()

    df = pd.concat(parts[name], ignore_index=True)

    key_cols = [c for c in df.columns if c not in ["weighted_sum", "weight_sum"]]

    out = (
        df.groupby(key_cols, dropna=False, as_index=False)
        .agg(
            weighted_sum=("weighted_sum", "sum"),
            weight_sum=("weight_sum", "sum")
        )
    )

    out = out[out["weight_sum"] > 0].copy()
    out[output_col] = out["weighted_sum"] / out["weight_sum"]

    return out


def choose_best_source(df, priority=("national", "state_sum", "area_sum", "all_rows")):
    if df.empty or "source_type" not in df.columns:
        return pd.DataFrame(), None

    for src in priority:
        sub = df[df["source_type"] == src].copy()
        if not sub.empty:
            return sub, src

    return pd.DataFrame(), None


def source_label(src):
    if src is None:
        return "unknown source"
    return src.replace("_", " ")


def save_table(df, rank, title):
    file_name = f"{rank}_{safe_name(title)}.csv"
    path = CFG["TABLE_DIR"] / file_name
    df.to_csv(path, index=False, encoding="utf-8")
    return path


def save_line_chart(df, rank, title, x_col, y_col, y_label):
    file_name = f"{rank}_{safe_name(title)}.png"
    path = CFG["CHART_DIR"] / file_name

    plot_df = df.sort_values(x_col).copy()

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(plot_df[x_col], plot_df[y_col], marker="o")

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel(y_label)
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_number))
    ax.grid(True, alpha=0.3)

    if plot_df[x_col].nunique() <= 20:
        ax.set_xticks(plot_df[x_col].dropna().astype(int).unique())

    plt.xticks(rotation=45)
    plt.tight_layout()
    fig.savefig(path, dpi=CFG["DPI"], bbox_inches="tight")
    plt.close(fig)

    return path


def save_barh_chart(df, rank, title, label_col, value_col, x_label):
    file_name = f"{rank}_{safe_name(title)}.png"
    path = CFG["CHART_DIR"] / file_name

    plot_df = df.sort_values(value_col, ascending=True).copy()

    height = max(6, len(plot_df) * 0.45 + 2)
    fig, ax = plt.subplots(figsize=(13, height))

    ax.barh(plot_df[label_col], plot_df[value_col])

    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_number))
    ax.grid(True, axis="x", alpha=0.3)

    plt.tight_layout()
    fig.savefig(path, dpi=CFG["DPI"], bbox_inches="tight")
    plt.close(fig)

    return path


def save_multi_line_chart(df, rank, title, x_col, series_col, y_col, y_label):
    file_name = f"{rank}_{safe_name(title)}.png"
    path = CFG["CHART_DIR"] / file_name

    pivot = (
        df.pivot_table(
            index=x_col,
            columns=series_col,
            values=y_col,
            aggfunc="sum"
        )
        .sort_index()
    )

    if pivot.empty:
        return None

    fig, ax = plt.subplots(figsize=(14, 7))

    for col in pivot.columns:
        ax.plot(pivot.index, pivot[col], marker="o", label=str(col))

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel(y_label)
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_number))
    ax.grid(True, alpha=0.3)

    if len(pivot.index) <= 20:
        ax.set_xticks(pivot.index.astype(int))

    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)

    plt.xticks(rotation=45)
    plt.tight_layout()
    fig.savefig(path, dpi=CFG["DPI"], bbox_inches="tight")
    plt.close(fig)

    return path


# ============================================================
# ACCUMULATORS
# ============================================================

sum_parts = defaultdict(list)
weighted_parts = defaultdict(list)

start_time = time.time()
rows_seen = 0


# ============================================================
# READ FILE IN CHUNKS
# ============================================================

print("\n" + "=" * 100)
print("READING + GROUPING IN CHUNKS")
print("=" * 100)

reader = pd.read_csv(
    FILE,
    usecols=USECOLS,
    dtype=DTYPE_MAP,
    chunksize=CFG["CHUNKSIZE"],
    encoding="utf-8",
    low_memory=False,
)

for chunk_num, df in enumerate(reader, start=1):
    chunk_start = time.time()

    # --------------------------------------------------------
    # Numeric conversion
    # --------------------------------------------------------
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["tot_emp"] = pd.to_numeric(df["tot_emp"], errors="coerce")
    df["a_median"] = pd.to_numeric(df["a_median"], errors="coerce")
    df["a_mean"] = pd.to_numeric(df["a_mean"], errors="coerce")
    df["h_median"] = pd.to_numeric(df["h_median"], errors="coerce")
    df["h_mean"] = pd.to_numeric(df["h_mean"], errors="coerce")
    df["area_type_num"] = pd.to_numeric(df["area_type"], errors="coerce")

    # --------------------------------------------------------
    # Text cleanup
    # --------------------------------------------------------
    df["_area_title"] = clean_text(df["area_title"])
    df["_naics_title"] = clean_text(df["naics_title"])
    df["_occ_title"] = clean_text(df["occ_title"])

    area_title_l = clean_lower(df["area_title"])
    data_level_l = clean_lower(df["data_level"])
    naics_title_l = clean_lower(df["naics_title"])
    occ_title_l = clean_lower(df["occ_title"])
    occ_code_s = clean_text(df["occ_code"])
    group_l = clean_lower(df["group"])

    state_label = clean_text(df["state_abbr"])
    prim_state = clean_text(df["prim_state"])
    state_name = clean_text(df["state"])

    state_label = state_label.mask(state_label == "", prim_state)
    state_label = state_label.mask(state_label == "", state_name)
    df["_state_label"] = state_label

    # --------------------------------------------------------
    # Main masks
    # --------------------------------------------------------
    m_year = df["year"].between(CFG["START_YEAR"], CFG["END_YEAR"], inclusive="both")
    m_emp = df["tot_emp"].notna()

    m_all_occ = (
        occ_code_s.eq("00-0000")
        | occ_title_l.eq("all occupations")
    )

    m_cross_industry = (
        df["_naics_title"].eq("")
        | naics_title_l.str.contains("cross-industry", na=False)
        | naics_title_l.str.contains("all industries", na=False)
        | clean_text(df["naics"]).isin(["0", "00", "000000", "0000000"])
    )

    m_has_industry = (
        ~m_cross_industry
        & df["_naics_title"].ne("")
    )

    m_national = (
        data_level_l.str.contains("national", na=False)
        | area_title_l.isin(["united states", "u.s.", "us", "national"])
        | df["area_type_num"].eq(1)
    )

    m_state_level = (
        (
            data_level_l.str.contains("state", na=False)
            & ~data_level_l.str.contains("metro|nonmetro|metropolitan|nonmetropolitan", na=False)
        )
        | df["area_type_num"].eq(2)
    )

    m_area_level = (
        data_level_l.str.contains("metro|nonmetro|metropolitan|nonmetropolitan", na=False)
        | df["area_type_num"].isin([3, 4, 5])
    )

    m_major_occ_group = (
        ~m_all_occ
        & (
            group_l.eq("major")
            | occ_code_s.str.match(r"^\d{2}-0000$", na=False)
        )
    )

    m_detailed_occ = (
        ~m_all_occ
        & ~m_major_occ_group
        & ~group_l.isin(["total", "major", "minor", "broad"])
        & df["_occ_title"].ne("")
    )

    source_masks = {
        "national": m_national,
        "state_sum": m_state_level,
        "area_sum": m_area_level,
        "all_rows": m_year,
    }

    # --------------------------------------------------------
    # Ranked chart data
    # --------------------------------------------------------

    for src, m_src in source_masks.items():
        base_cross = m_year & m_src & m_cross_industry & m_emp

        # 1. Overall employment by year
        add_sum(
            sum_parts,
            "overall_emp",
            df,
            base_cross & m_all_occ,
            ["year"],
            "tot_emp",
            src
        )

        # 2. Annual median wage by year
        add_weighted(
            weighted_parts,
            "annual_wage_year",
            df,
            base_cross & m_all_occ,
            ["year"],
            "a_median",
            "tot_emp",
            src
        )

        # 3. Occupation groups by employment
        add_sum(
            sum_parts,
            "occ_group_emp",
            df,
            base_cross & m_major_occ_group,
            ["year", "_occ_title"],
            "tot_emp",
            src
        )

        # 4. Detailed occupations by employment
        add_sum(
            sum_parts,
            "detail_occ_emp",
            df,
            base_cross & m_detailed_occ,
            ["year", "_occ_title"],
            "tot_emp",
            src
        )

        # 5. Occupation group annual median wage
        add_weighted(
            weighted_parts,
            "occ_group_annual_wage",
            df,
            base_cross & m_major_occ_group,
            ["year", "_occ_title"],
            "a_median",
            "tot_emp",
            src
        )

        # 6. Detailed occupation annual median wage
        add_weighted(
            weighted_parts,
            "detail_occ_annual_wage",
            df,
            base_cross & m_detailed_occ,
            ["year", "_occ_title"],
            "a_median",
            "tot_emp",
            src
        )

        # 7. Hourly median wage by year
        add_weighted(
            weighted_parts,
            "hourly_wage_year",
            df,
            base_cross & m_all_occ,
            ["year"],
            "h_median",
            "tot_emp",
            src
        )

        # 8. Industry employment
        add_sum(
            sum_parts,
            "industry_emp",
            df,
            m_year & m_src & m_has_industry & m_all_occ & m_emp,
            ["year", "_naics_title"],
            "tot_emp",
            src
        )

    # 9. State employment chart
    add_sum(
        sum_parts,
        "state_emp",
        df,
        m_year & m_state_level & m_cross_industry & m_all_occ & m_emp & df["_state_label"].ne(""),
        ["year", "_state_label"],
        "tot_emp",
        "state"
    )

    # 10. Metro / nonmetro area employment chart
    add_sum(
        sum_parts,
        "area_emp",
        df,
        m_year & m_area_level & m_cross_industry & m_all_occ & m_emp & df["_area_title"].ne(""),
        ["year", "_area_title"],
        "tot_emp",
        "area"
    )

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------
    rows_seen += len(df)
    elapsed = time.time() - start_time
    speed = rows_seen / elapsed if elapsed > 0 else 0
    pct = rows_seen / CFG["EXPECTED_ROWS"] * 100 if CFG["EXPECTED_ROWS"] else 0

    if chunk_num % CFG["PRINT_EVERY_CHUNK"] == 0:
        print(
            f"Chunk {chunk_num:,} done | "
            f"Rows scanned: {rows_seen:,} / {CFG['EXPECTED_ROWS']:,} | "
            f"{pct:,.2f}% | "
            f"Speed: {speed:,.0f} rows/sec | "
            f"Chunk time: {time.time() - chunk_start:.2f} sec",
            flush=True
        )

    del df
    if chunk_num % 5 == 0:
        gc.collect()


print("\n" + "=" * 100)
print("CHUNK READING COMPLETE")
print("=" * 100)
print(f"Rows scanned: {rows_seen:,}")
print(f"Time elapsed: {time.time() - start_time:,.2f} seconds")


# ============================================================
# FINALIZE SMALL SUMMARY TABLES
# ============================================================

print("\n" + "=" * 100)
print("FINALIZING SUMMARY TABLES")
print("=" * 100)

overall_emp = finalize_sum(sum_parts, "overall_emp", "tot_emp")
occ_group_emp = finalize_sum(sum_parts, "occ_group_emp", "tot_emp")
detail_occ_emp = finalize_sum(sum_parts, "detail_occ_emp", "tot_emp")
state_emp = finalize_sum(sum_parts, "state_emp", "tot_emp")
area_emp = finalize_sum(sum_parts, "area_emp", "tot_emp")
industry_emp = finalize_sum(sum_parts, "industry_emp", "tot_emp")

annual_wage_year = finalize_weighted(weighted_parts, "annual_wage_year", "annual_median_wage")
hourly_wage_year = finalize_weighted(weighted_parts, "hourly_wage_year", "hourly_median_wage")
occ_group_annual_wage = finalize_weighted(weighted_parts, "occ_group_annual_wage", "annual_median_wage")
detail_occ_annual_wage = finalize_weighted(weighted_parts, "detail_occ_annual_wage", "annual_median_wage")

print("Summary tables ready.")


# ============================================================
# CREATE RANKED OUTPUTS
# ============================================================

generated = []
rank = 1
top_occ_group_names = []
top_detail_occ_names = []


def register_output(title, table_df, chart_path, table_path):
    generated.append({
        "rank": len(generated) + 1,
        "title": title,
        "chart_file": str(chart_path),
        "table_file": str(table_path),
    })


def skip(title, reason):
    print(f"SKIPPED: {title} | {reason}")


# ------------------------------------------------------------
# 1. Overall job count by year
# ------------------------------------------------------------

title = "Overall Job Count By Year"
df_src, src = choose_best_source(overall_emp)

if df_src.empty:
    skip(title, "No data found.")
else:
    table = (
        df_src.groupby("year", as_index=False)["tot_emp"]
        .sum()
        .sort_values("year")
        .rename(columns={"tot_emp": "total_employment"})
    )
    table["source_used"] = source_label(src)

    table_path = save_table(table, rank, title)
    chart_path = save_line_chart(
        table,
        rank,
        title,
        "year",
        "total_employment",
        "Total employment"
    )
    register_output(title, table, chart_path, table_path)
    print(f"SAVED {rank}: {title}")
    rank += 1


# ------------------------------------------------------------
# 2. Annual median wage by year
# ------------------------------------------------------------

title = "Annual Median Wage By Year"
df_src, src = choose_best_source(annual_wage_year)

if df_src.empty:
    skip(title, "No data found.")
else:
    table = (
        df_src.groupby("year", as_index=False)
        .agg(
            weighted_sum=("weighted_sum", "sum"),
            weight_sum=("weight_sum", "sum")
        )
        .sort_values("year")
    )
    table["annual_median_wage"] = table["weighted_sum"] / table["weight_sum"]
    table = table[["year", "annual_median_wage", "weight_sum"]]
    table = table.rename(columns={"weight_sum": "employment_used_for_weight"})
    table["source_used"] = source_label(src)

    table_path = save_table(table, rank, title)
    chart_path = save_line_chart(
        table,
        rank,
        title,
        "year",
        "annual_median_wage",
        "Annual median wage"
    )
    register_output(title, table, chart_path, table_path)
    print(f"SAVED {rank}: {title}")
    rank += 1


# ------------------------------------------------------------
# 3. Top occupation groups by employment, latest year
# ------------------------------------------------------------

title = "Top 12 Occupation Groups By Employment"
df_src, src = choose_best_source(occ_group_emp)

if df_src.empty:
    skip(title, "No data found.")
else:
    latest_year = int(df_src["year"].max())

    table = (
        df_src[df_src["year"] == latest_year]
        .groupby("_occ_title", as_index=False)["tot_emp"]
        .sum()
        .sort_values("tot_emp", ascending=False)
        .head(CFG["TOP_N"])
        .rename(columns={
            "_occ_title": "occupation_group",
            "tot_emp": "total_employment"
        })
    )

    table.insert(0, "year", latest_year)
    table["source_used"] = source_label(src)

    top_occ_group_names = table["occupation_group"].tolist()

    table_path = save_table(table, rank, title)
    chart_path = save_barh_chart(
        table,
        rank,
        title,
        "occupation_group",
        "total_employment",
        "Total employment"
    )
    register_output(title, table, chart_path, table_path)
    print(f"SAVED {rank}: {title}")
    rank += 1


# ------------------------------------------------------------
# 4. Top detailed occupations by employment, latest year
# ------------------------------------------------------------

title = "Top 12 Detailed Occupations By Employment"
df_src, src = choose_best_source(detail_occ_emp)

if df_src.empty:
    skip(title, "No data found.")
else:
    latest_year = int(df_src["year"].max())

    table = (
        df_src[df_src["year"] == latest_year]
        .groupby("_occ_title", as_index=False)["tot_emp"]
        .sum()
        .sort_values("tot_emp", ascending=False)
        .head(CFG["TOP_N"])
        .rename(columns={
            "_occ_title": "occupation",
            "tot_emp": "total_employment"
        })
    )

    table.insert(0, "year", latest_year)
    table["source_used"] = source_label(src)

    top_detail_occ_names = table["occupation"].tolist()

    table_path = save_table(table, rank, title)
    chart_path = save_barh_chart(
        table,
        rank,
        title,
        "occupation",
        "total_employment",
        "Total employment"
    )
    register_output(title, table, chart_path, table_path)
    print(f"SAVED {rank}: {title}")
    rank += 1


# ------------------------------------------------------------
# 5. Employment trend for top occupation groups
# ------------------------------------------------------------

title = "Employment Trend For Top 12 Occupation Groups"
df_src, src = choose_best_source(occ_group_emp)

if df_src.empty or len(top_occ_group_names) == 0:
    skip(title, "No top occupation groups found.")
else:
    table = (
        df_src[df_src["_occ_title"].isin(top_occ_group_names)]
        .groupby(["year", "_occ_title"], as_index=False)["tot_emp"]
        .sum()
        .rename(columns={
            "_occ_title": "occupation_group",
            "tot_emp": "total_employment"
        })
        .sort_values(["occupation_group", "year"])
    )

    table["source_used"] = source_label(src)

    table_path = save_table(table, rank, title)
    chart_path = save_multi_line_chart(
        table,
        rank,
        title,
        "year",
        "occupation_group",
        "total_employment",
        "Total employment"
    )

    if chart_path is None:
        skip(title, "Could not create chart.")
    else:
        register_output(title, table, chart_path, table_path)
        print(f"SAVED {rank}: {title}")
        rank += 1


# ------------------------------------------------------------
# 6. Annual median wage trend for top occupation groups
# ------------------------------------------------------------

title = "Annual Median Wage Trend For Top 12 Occupation Groups"
df_src, src = choose_best_source(occ_group_annual_wage)

if df_src.empty or len(top_occ_group_names) == 0:
    skip(title, "No wage data for top occupation groups.")
else:
    table = df_src[df_src["_occ_title"].isin(top_occ_group_names)].copy()

    table = (
        table.groupby(["year", "_occ_title"], as_index=False)
        .agg(
            weighted_sum=("weighted_sum", "sum"),
            weight_sum=("weight_sum", "sum")
        )
    )

    table["annual_median_wage"] = table["weighted_sum"] / table["weight_sum"]

    table = (
        table.rename(columns={
            "_occ_title": "occupation_group",
            "weight_sum": "employment_used_for_weight"
        })
        .sort_values(["occupation_group", "year"])
    )

    table = table[[
        "year",
        "occupation_group",
        "annual_median_wage",
        "employment_used_for_weight"
    ]]

    table["source_used"] = source_label(src)

    table_path = save_table(table, rank, title)
    chart_path = save_multi_line_chart(
        table,
        rank,
        title,
        "year",
        "occupation_group",
        "annual_median_wage",
        "Annual median wage"
    )

    if chart_path is None:
        skip(title, "Could not create chart.")
    else:
        register_output(title, table, chart_path, table_path)
        print(f"SAVED {rank}: {title}")
        rank += 1


# ------------------------------------------------------------
# 7. Top detailed occupations by annual median wage
# ------------------------------------------------------------

title = "Top 12 Detailed Occupations By Annual Median Wage"
df_src, src = choose_best_source(detail_occ_annual_wage)

if df_src.empty:
    skip(title, "No wage data found.")
else:
    latest_year = int(df_src["year"].max())

    temp = df_src[df_src["year"] == latest_year].copy()

    temp = (
        temp.groupby("_occ_title", as_index=False)
        .agg(
            weighted_sum=("weighted_sum", "sum"),
            weight_sum=("weight_sum", "sum")
        )
    )

    temp = temp[temp["weight_sum"] >= CFG["MIN_EMP_FOR_WAGE_TOP"]].copy()
    temp["annual_median_wage"] = temp["weighted_sum"] / temp["weight_sum"]

    table = (
        temp.sort_values("annual_median_wage", ascending=False)
        .head(CFG["TOP_N"])
        .rename(columns={
            "_occ_title": "occupation",
            "weight_sum": "employment_used_for_weight"
        })
    )

    table.insert(0, "year", latest_year)
    table = table[[
        "year",
        "occupation",
        "annual_median_wage",
        "employment_used_for_weight"
    ]]
    table["source_used"] = source_label(src)

    table_path = save_table(table, rank, title)
    chart_path = save_barh_chart(
        table,
        rank,
        title,
        "occupation",
        "annual_median_wage",
        "Annual median wage"
    )
    register_output(title, table, chart_path, table_path)
    print(f"SAVED {rank}: {title}")
    rank += 1


# ------------------------------------------------------------
# 8. Top states by employment, latest year
# ------------------------------------------------------------

title = "Top 12 States By Employment"

if state_emp.empty:
    skip(title, "No state-level data found.")
else:
    latest_year = int(state_emp["year"].max())

    table = (
        state_emp[state_emp["year"] == latest_year]
        .groupby("_state_label", as_index=False)["tot_emp"]
        .sum()
        .sort_values("tot_emp", ascending=False)
        .head(CFG["TOP_N"])
        .rename(columns={
            "_state_label": "state",
            "tot_emp": "total_employment"
        })
    )

    table.insert(0, "year", latest_year)
    table["source_used"] = "state"

    table_path = save_table(table, rank, title)
    chart_path = save_barh_chart(
        table,
        rank,
        title,
        "state",
        "total_employment",
        "Total employment"
    )
    register_output(title, table, chart_path, table_path)
    print(f"SAVED {rank}: {title}")
    rank += 1


# ------------------------------------------------------------
# 9. Top industries by employment, latest year
# ------------------------------------------------------------

title = "Top 12 Industries By Employment"
df_src, src = choose_best_source(industry_emp)

if df_src.empty:
    skip(title, "No industry data found.")
else:
    latest_year = int(df_src["year"].max())

    table = (
        df_src[df_src["year"] == latest_year]
        .groupby("_naics_title", as_index=False)["tot_emp"]
        .sum()
        .sort_values("tot_emp", ascending=False)
        .head(CFG["TOP_N"])
        .rename(columns={
            "_naics_title": "industry",
            "tot_emp": "total_employment"
        })
    )

    table.insert(0, "year", latest_year)
    table["source_used"] = source_label(src)

    table_path = save_table(table, rank, title)
    chart_path = save_barh_chart(
        table,
        rank,
        title,
        "industry",
        "total_employment",
        "Total employment"
    )
    register_output(title, table, chart_path, table_path)
    print(f"SAVED {rank}: {title}")
    rank += 1


# ------------------------------------------------------------
# 10. Top metro / nonmetro areas by employment
# ------------------------------------------------------------

title = "Top 12 Areas By Employment"

if area_emp.empty:
    skip(title, "No metro/nonmetro area data found.")
else:
    latest_year = int(area_emp["year"].max())

    table = (
        area_emp[area_emp["year"] == latest_year]
        .groupby("_area_title", as_index=False)["tot_emp"]
        .sum()
        .sort_values("tot_emp", ascending=False)
        .head(CFG["TOP_N"])
        .rename(columns={
            "_area_title": "area",
            "tot_emp": "total_employment"
        })
    )

    table.insert(0, "year", latest_year)
    table["source_used"] = "area"

    table_path = save_table(table, rank, title)
    chart_path = save_barh_chart(
        table,
        rank,
        title,
        "area",
        "total_employment",
        "Total employment"
    )
    register_output(title, table, chart_path, table_path)
    print(f"SAVED {rank}: {title}")
    rank += 1


# ------------------------------------------------------------
# 11. Hourly median wage by year
# ------------------------------------------------------------

title = "Hourly Median Wage By Year"
df_src, src = choose_best_source(hourly_wage_year)

if df_src.empty:
    skip(title, "No hourly wage data found.")
else:
    table = (
        df_src.groupby("year", as_index=False)
        .agg(
            weighted_sum=("weighted_sum", "sum"),
            weight_sum=("weight_sum", "sum")
        )
        .sort_values("year")
    )

    table["hourly_median_wage"] = table["weighted_sum"] / table["weight_sum"]

    table = table[["year", "hourly_median_wage", "weight_sum"]]
    table = table.rename(columns={"weight_sum": "employment_used_for_weight"})
    table["source_used"] = source_label(src)

    table_path = save_table(table, rank, title)
    chart_path = save_line_chart(
        table,
        rank,
        title,
        "year",
        "hourly_median_wage",
        "Hourly median wage"
    )
    register_output(title, table, chart_path, table_path)
    print(f"SAVED {rank}: {title}")
    rank += 1


# ============================================================
# SAVE INDEX FILE
# ============================================================

index_df = pd.DataFrame(generated)

index_path = CFG["TABLE_DIR"] / "00_OUTPUT_INDEX.csv"
index_df.to_csv(index_path, index=False, encoding="utf-8")

print("\n" + "=" * 100)
print("FINAL OUTPUT")
print("=" * 100)
print(f"Charts saved in: {CFG['CHART_DIR']}")
print(f"Tables saved in: {CFG['TABLE_DIR']}")
print(f"Index saved as : {index_path}")
print(f"Charts created : {len(generated)}")

print("\nOUTPUT INDEX:")
print(index_df.to_string(index=False))

print("\nDONE.")
print("This code used chunk reading and did NOT load the full CSV into memory.")

# Check for error

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc
import re

# ============================================================
# JOB DATA DOUBLE CHECK ONLY
# NO SAVE
# NO FOLDER
# NO CSV OUTPUT
# NO PNG OUTPUT
# NO CACHE FILE
#
# MEMORY OPTIMIZED:
# - Reads CSV in chunks
# - Keeps only grouped summary in memory
# - Does not load full file at once
# ============================================================

INPUT_FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_1.csv"

START_YEAR = 2010
END_YEAR = 2025
TOP_N = 12
CHUNKSIZE = 150_000

# Best for national chart check
FILTER_US_TOTAL_ONLY = True
FILTER_ALL_INDUSTRIES_ONLY = True
FILTER_ALL_OWNERSHIP_ONLY = True

# ============================================================
# OCCUPATION MAJOR GROUP NAMES
# ============================================================

MAJOR_CODE_NAME = {
    "11": "Management Occupations",
    "13": "Business and Financial Operations Occupations",
    "15": "Computer and Mathematical Occupations",
    "17": "Architecture and Engineering Occupations",
    "19": "Life, Physical, and Social Science Occupations",
    "21": "Community and Social Service Occupations",
    "23": "Legal Occupations",
    "25": "Educational Instruction and Library Occupations",
    "27": "Arts, Design, Entertainment, Sports, and Media Occupations",
    "29": "Healthcare Practitioners and Technical Occupations",
    "31": "Healthcare Support Occupations",
    "33": "Protective Service Occupations",
    "35": "Food Preparation and Serving Related Occupations",
    "37": "Building and Grounds Cleaning and Maintenance Occupations",
    "39": "Personal Care and Service Occupations",
    "41": "Sales and Related Occupations",
    "43": "Office and Administrative Support Occupations",
    "45": "Farming, Fishing, and Forestry Occupations",
    "47": "Construction and Extraction Occupations",
    "49": "Installation, Maintenance, and Repair Occupations",
    "51": "Production Occupations",
    "53": "Transportation and Material Moving Occupations",
}

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_col_name(x):
    return str(x).strip().lower()


def find_col(columns, choices, required=True):
    lower_map = {clean_col_name(c): c for c in columns}

    for choice in choices:
        key = clean_col_name(choice)
        if key in lower_map:
            return lower_map[key]

    if required:
        raise ValueError(f"Missing required column. Tried: {choices}")

    return None


def normalize_occ_code(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(".0", "", regex=False)
    )


def get_major_code_from_occ_code(series):
    return series.astype("string").str.extract(r"^(\d{2})", expand=False)


def is_major_group_occ_code(series):
    s = normalize_occ_code(series)
    return s.str.match(r"^\d{2}-0000$", na=False) & ~s.str.startswith("00", na=False)


def safe_numeric(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False),
        errors="coerce"
    )


def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


# ============================================================
# CHECK INPUT ONLY
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File not found:\n{INPUT_FILE}")

print_section("CHECK ONLY STARTED")
print(f"Input file: {INPUT_FILE}")
print("Save file: NO")
print("Create folder: NO")
print("Create chart PNG: NO")
print("Create cache: NO")

# ============================================================
# READ HEADER ONLY
# ============================================================

header = pd.read_csv(INPUT_FILE, nrows=0)
columns = list(header.columns)

YEAR_COL = find_col(columns, ["year"])
OCC_CODE_COL = find_col(columns, ["occ_code", "OCC_CODE"])
OCC_TITLE_COL = find_col(columns, ["occ_title", "OCC_TITLE"], required=False)
O_GROUP_COL = find_col(columns, ["o_group", "GROUP", "group"], required=False)

TOT_EMP_COL = find_col(columns, ["tot_emp", "TOT_EMP", "total_employment"])
A_MEDIAN_COL = find_col(columns, ["a_median", "A_MEDIAN", "annual_median_wage", "annual_median"], required=False)

AREA_COL = find_col(columns, ["area", "AREA"], required=False)
AREA_TITLE_COL = find_col(columns, ["area_title", "AREA_TITLE"], required=False)
AREA_TYPE_COL = find_col(columns, ["area_type", "AREA_TYPE"], required=False)

NAICS_COL = find_col(columns, ["naics", "NAICS"], required=False)
NAICS_TITLE_COL = find_col(columns, ["naics_title", "NAICS_TITLE"], required=False)

OWN_CODE_COL = find_col(columns, ["own_code", "OWN_CODE"], required=False)

usecols = [YEAR_COL, OCC_CODE_COL, TOT_EMP_COL]

for c in [
    OCC_TITLE_COL,
    O_GROUP_COL,
    A_MEDIAN_COL,
    AREA_COL,
    AREA_TITLE_COL,
    AREA_TYPE_COL,
    NAICS_COL,
    NAICS_TITLE_COL,
    OWN_CODE_COL,
]:
    if c is not None and c not in usecols:
        usecols.append(c)

print_section("COLUMNS USED")
for c in usecols:
    print("-", c)

# ============================================================
# READ IN CHUNKS
# ============================================================

start_time = time.time()

summary_parts = []
title_parts = []
area_audit_parts = []

total_rows_seen = 0
total_rows_after_year = 0
total_rows_major_group = 0
total_rows_after_filters = 0

reader = pd.read_csv(
    INPUT_FILE,
    usecols=usecols,
    chunksize=CHUNKSIZE,
    low_memory=False
)

for chunk_number, chunk in enumerate(reader, start=1):
    chunk_start = time.time()

    total_rows_seen += len(chunk)

    rename_map = {
        YEAR_COL: "year",
        OCC_CODE_COL: "occ_code",
        TOT_EMP_COL: "tot_emp",
    }

    if OCC_TITLE_COL:
        rename_map[OCC_TITLE_COL] = "occ_title"
    if O_GROUP_COL:
        rename_map[O_GROUP_COL] = "o_group"
    if A_MEDIAN_COL:
        rename_map[A_MEDIAN_COL] = "a_median"
    if AREA_COL:
        rename_map[AREA_COL] = "area"
    if AREA_TITLE_COL:
        rename_map[AREA_TITLE_COL] = "area_title"
    if AREA_TYPE_COL:
        rename_map[AREA_TYPE_COL] = "area_type"
    if NAICS_COL:
        rename_map[NAICS_COL] = "naics"
    if NAICS_TITLE_COL:
        rename_map[NAICS_TITLE_COL] = "naics_title"
    if OWN_CODE_COL:
        rename_map[OWN_CODE_COL] = "own_code"

    chunk = chunk.rename(columns=rename_map)

    # Year filter
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk = chunk[(chunk["year"] >= START_YEAR) & (chunk["year"] <= END_YEAR)].copy()
    total_rows_after_year += len(chunk)

    if len(chunk) == 0:
        continue

    chunk["year"] = chunk["year"].astype("Int64")

    # Occupation code normalization
    chunk["occ_code"] = normalize_occ_code(chunk["occ_code"])
    chunk["major_code"] = get_major_code_from_occ_code(chunk["occ_code"])

    # Keep major occupation groups only
    chunk = chunk[is_major_group_occ_code(chunk["occ_code"])].copy()
    chunk = chunk[chunk["major_code"].isin(MAJOR_CODE_NAME.keys())].copy()

    total_rows_major_group += len(chunk)

    if len(chunk) == 0:
        continue

    # Numeric conversion
    chunk["tot_emp"] = safe_numeric(chunk["tot_emp"])

    if "a_median" in chunk.columns:
        chunk["a_median"] = safe_numeric(chunk["a_median"])
    else:
        chunk["a_median"] = np.nan

    # Area audit BEFORE filters
    temp_area = chunk.copy()

    if "area_title" not in temp_area.columns:
        temp_area["area_title"] = "NO_AREA_TITLE_COLUMN"

    if "area_type" not in temp_area.columns:
        temp_area["area_type"] = "NO_AREA_TYPE_COLUMN"

    area_audit = (
        temp_area
        .groupby(["year", "area_type", "area_title"], dropna=False)
        .agg(
            rows=("occ_code", "size"),
            employment_sum=("tot_emp", "sum")
        )
        .reset_index()
    )

    area_audit_parts.append(area_audit)

    # US total filter
    if FILTER_US_TOTAL_ONLY:
        us_mask = pd.Series(False, index=chunk.index)

        if "area" in chunk.columns:
            area_s = chunk["area"].astype("string").str.strip()
            us_mask = us_mask | area_s.isin(["99", "0000000", "0"])

        if "area_title" in chunk.columns:
            area_title_s = chunk["area_title"].astype("string").str.strip().str.lower()
            us_mask = us_mask | area_title_s.isin([
                "u.s.",
                "us",
                "united states",
                "national"
            ])

        if "area_type" in chunk.columns:
            area_type_s = chunk["area_type"].astype("string").str.strip()
            us_mask = us_mask | area_type_s.isin(["1"])

        chunk = chunk[us_mask].copy()

    # All industries filter
    if FILTER_ALL_INDUSTRIES_ONLY:
        industry_mask = pd.Series(True, index=chunk.index)

        if "naics" in chunk.columns:
            naics_s = (
                chunk["naics"]
                .astype("string")
                .str.strip()
                .str.replace(".0", "", regex=False)
            )

            industry_mask = naics_s.isin([
                "0",
                "00",
                "000",
                "0000",
                "00000",
                "000000",
                "0000000",
                "00-000000"
            ])

        if "naics_title" in chunk.columns:
            title_s = chunk["naics_title"].astype("string").str.strip().str.lower()
            title_mask = title_s.str.contains(
                "all industries|cross-industry|cross industry",
                na=False
            )

            industry_mask = industry_mask | title_mask

        chunk = chunk[industry_mask].copy()

    # All ownership filter
    if FILTER_ALL_OWNERSHIP_ONLY and "own_code" in chunk.columns:
        own_s = (
            chunk["own_code"]
            .astype("string")
            .str.strip()
            .str.replace(".0", "", regex=False)
        )

        own_mask = own_s.isin(["1235", "0", "00"])

        if own_mask.sum() > 0:
            chunk = chunk[own_mask].copy()

    total_rows_after_filters += len(chunk)

    if len(chunk) == 0:
        continue

    chunk["occupation_group"] = chunk["major_code"].map(MAJOR_CODE_NAME)

    # Title variants check
    if "occ_title" in chunk.columns:
        title_temp = (
            chunk[["year", "major_code", "occ_code", "occ_title"]]
            .drop_duplicates()
            .copy()
        )
        title_parts.append(title_temp)

    # Main summary
    grouped = (
        chunk
        .groupby(["year", "major_code", "occupation_group"], dropna=False)
        .agg(
            row_count=("occ_code", "size"),
            total_employment=("tot_emp", "sum"),
            annual_median_wage=("a_median", "mean"),
            nonmissing_employment=("tot_emp", "count"),
            nonmissing_wage=("a_median", "count")
        )
        .reset_index()
    )

    summary_parts.append(grouped)

    elapsed = time.time() - start_time
    chunk_time = time.time() - chunk_start

    print(
        f"Chunk {chunk_number:,} done | "
        f"rows seen: {total_rows_seen:,} | "
        f"rows kept: {total_rows_after_filters:,} | "
        f"chunk sec: {chunk_time:,.1f} | "
        f"total min: {elapsed / 60:,.1f}",
        flush=True
    )

    del chunk
    gc.collect()

# ============================================================
# COMBINE SMALL SUMMARY ONLY
# ============================================================

if not summary_parts:
    raise ValueError("No rows kept after filters. Try FILTER_US_TOTAL_ONLY = False to inspect data.")

summary = pd.concat(summary_parts, ignore_index=True)

summary = (
    summary
    .groupby(["year", "major_code", "occupation_group"], dropna=False)
    .agg(
        row_count=("row_count", "sum"),
        total_employment=("total_employment", "sum"),
        annual_median_wage=("annual_median_wage", "mean"),
        nonmissing_employment=("nonmissing_employment", "sum"),
        nonmissing_wage=("nonmissing_wage", "sum")
    )
    .reset_index()
)

# ============================================================
# PRINT SUMMARY
# ============================================================

print_section("ROW CHECK")
print(f"Total rows seen:              {total_rows_seen:,}")
print(f"Rows after year filter:       {total_rows_after_year:,}")
print(f"Rows major occupation groups: {total_rows_major_group:,}")
print(f"Rows after final filters:     {total_rows_after_filters:,}")

print_section("YEARS FOUND AFTER FILTERS")
years_found = sorted(summary["year"].dropna().astype(int).unique().tolist())
print(years_found)

missing_years = [y for y in range(START_YEAR, END_YEAR + 1) if y not in years_found]

if missing_years:
    print("Missing years:", missing_years)
else:
    print("No missing years from", START_YEAR, "to", END_YEAR)

# ============================================================
# TOP 12 BY LATEST YEAR
# ============================================================

latest_year = int(summary["year"].max())

top_table = (
    summary[summary["year"] == latest_year]
    .sort_values("total_employment", ascending=False)
    .head(TOP_N)
    .copy()
    .reset_index(drop=True)
)

top_table.insert(0, "rank", top_table.index + 1)

top_codes = top_table["major_code"].tolist()

print_section(f"TOP {TOP_N} OCCUPATION GROUPS BY {latest_year} EMPLOYMENT")
print(
    top_table[
        [
            "rank",
            "major_code",
            "occupation_group",
            "total_employment",
            "annual_median_wage",
            "row_count"
        ]
    ].to_string(index=False)
)

# ============================================================
# MISSING CHECK FOR TOP 12
# ============================================================

all_years = list(range(START_YEAR, END_YEAR + 1))

full_index = pd.MultiIndex.from_product(
    [all_years, top_codes],
    names=["year", "major_code"]
).to_frame(index=False)

top_summary = summary[summary["major_code"].isin(top_codes)].copy()

missing_check = full_index.merge(
    top_summary,
    on=["year", "major_code"],
    how="left"
)

missing_check["occupation_group"] = missing_check["major_code"].map(MAJOR_CODE_NAME)

missing_check["employment_missing"] = (
    missing_check["total_employment"].isna() |
    (missing_check["nonmissing_employment"].fillna(0) == 0)
)

missing_check["wage_missing"] = (
    missing_check["annual_median_wage"].isna() |
    (missing_check["nonmissing_wage"].fillna(0) == 0)
)

employment_missing = missing_check[missing_check["employment_missing"]].copy()
wage_missing = missing_check[missing_check["wage_missing"]].copy()

print_section("MISSING EMPLOYMENT CHECK FOR TOP 12")

if employment_missing.empty:
    print("No missing employment values for Top 12.")
else:
    print(
        employment_missing[
            ["year", "major_code", "occupation_group"]
        ].to_string(index=False)
    )

print_section("MISSING ANNUAL MEDIAN WAGE CHECK FOR TOP 12")

if wage_missing.empty:
    print("No missing annual median wage values for Top 12.")
else:
    print(
        wage_missing[
            ["year", "major_code", "occupation_group"]
        ].to_string(index=False)
    )

# ============================================================
# TITLE VARIANT CHECK
# ============================================================

print_section("TITLE VARIANT CHECK")

if title_parts:
    title_variants = pd.concat(title_parts, ignore_index=True).drop_duplicates()

    title_variants_top12 = (
        title_variants[title_variants["major_code"].isin(top_codes)]
        .sort_values(["major_code", "year", "occ_title"])
        .copy()
    )

    if title_variants_top12.empty:
        print("No title variants found for Top 12.")
    else:
        print(
            title_variants_top12[
                ["year", "major_code", "occ_code", "occ_title"]
            ].to_string(index=False)
        )
else:
    print("No occ_title column found, so title variant check skipped.")

# ============================================================
# DUPLICATE CHECK AFTER FILTERS
# ============================================================

duplicates = top_summary[top_summary["row_count"] > 1].copy()

print_section("DUPLICATE CHECK AFTER FILTERS")

if duplicates.empty:
    print("No duplicate year/group rows after filters.")
else:
    print("Some year/group combinations have more than 1 row.")
    print("This can cause double counting if the file has multiple releases or duplicate area/industry rows.")
    print(
        duplicates[
            ["year", "major_code", "occupation_group", "row_count", "total_employment"]
        ].sort_values(["year", "major_code"]).to_string(index=False)
    )

# ============================================================
# AREA DOUBLE COUNT CHECK
# ============================================================

print_section("AREA DOUBLE COUNT CHECK")

if area_audit_parts:
    area_audit_all = pd.concat(area_audit_parts, ignore_index=True)

    area_audit_all = (
        area_audit_all
        .groupby(["year", "area_type", "area_title"], dropna=False)
        .agg(
            rows=("rows", "sum"),
            employment_sum=("employment_sum", "sum")
        )
        .reset_index()
    )

    latest_area_audit = (
        area_audit_all[area_audit_all["year"] == latest_year]
        .sort_values("employment_sum", ascending=False)
        .head(25)
    )

    print(f"Top area totals for {latest_year} before US-only filter:")
    print(
        latest_area_audit[
            ["year", "area_type", "area_title", "rows", "employment_sum"]
        ].to_string(index=False)
    )
else:
    print("Area audit skipped because area columns were not available.")

# ============================================================
# FINAL RESULT
# ============================================================

elapsed = time.time() - start_time

print_section("DONE - CHECK ONLY")
print(f"Runtime minutes: {elapsed / 60:,.2f}")
print("Files created: 0")
print("Folders created: 0")
print("Original CSV changed: NO")

print("\nMost important result to check:")
print("1. Missing employment check")
print("2. Missing annual median wage check")
print("3. Title variant check")
print("4. Duplicate check after filters")
print("5. Area double count check")

# Fix code chart

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import re
import gc
import time
import textwrap

# ============================================================
# JOB CHART 2 + JOB TABLE 2
# FIXED VERSION
#
# INPUT:
# /Downloads/all_2010-2025_job_CLEANED_numeric_only_1.csv
#
# OUTPUT:
# /Downloads/Job_Chart_2
# /Downloads/Job_Table_2
#
# FIXES:
# 1. Uses U.S. only
# 2. Avoids state/metro/area double counting
# 3. Uses major occupation code, not title only
# 4. Fixes title variant problem
# 5. Deduplicates year + major_code, especially 2012
# 6. Starts at 2011 because 2010 is missing after filters
# 7. Memory optimized with chunk reading
# ============================================================

# ============================================================
# SETTINGS
# ============================================================

INPUT_FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_1.csv"

CHART_DIR = Path.home() / "Downloads" / "Job_Chart_2"
TABLE_DIR = Path.home() / "Downloads" / "Job_Table_2"

CHART_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2011
END_YEAR = 2025
TOP_N = 12
CHUNKSIZE = 150_000

# If later you fix 2010 in the source file, change START_YEAR to 2010.

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_num(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan}),
        errors="coerce"
    )


def normalize_text(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )


def safe_filename(name):
    name = re.sub(r"[^\w\s\-]", "", str(name))
    name = re.sub(r"\s+", "_", name.strip())
    return name[:160]


def fmt_millions(x, pos=None):
    try:
        return f"{x / 1_000_000:.1f}M"
    except Exception:
        return ""


def fmt_number(x, pos=None):
    try:
        return f"{int(x):,}"
    except Exception:
        return ""


def fmt_dollar_k(x, pos=None):
    try:
        return f"${x / 1000:.0f}k"
    except Exception:
        return ""


def save_table(rank, title, df):
    path = TABLE_DIR / f"{rank}_{safe_filename(title)}.csv"
    df.to_csv(path, index=False)
    print(f"Saved table {rank}: {path}", flush=True)
    return path


def save_chart(rank, title):
    path = CHART_DIR / f"{rank}_{safe_filename(title)}.png"
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"Saved chart {rank}: {path}", flush=True)
    return path


def wrap_labels(labels, width=28):
    return ["\n".join(textwrap.wrap(str(x), width=width)) for x in labels]


# Short chart labels by SOC major code
SHORT_NAMES = {
    "11": "Management",
    "13": "Business / Financial",
    "15": "Computer / Math",
    "17": "Architecture / Engineering",
    "19": "Life / Physical / Social Science",
    "21": "Community / Social Service",
    "23": "Legal",
    "25": "Education / Library",
    "27": "Arts / Media",
    "29": "Healthcare Practitioners",
    "31": "Healthcare Support",
    "33": "Protective Service",
    "35": "Food Serving",
    "37": "Building / Grounds",
    "39": "Personal Care / Service",
    "41": "Sales",
    "43": "Office / Admin Support",
    "45": "Farming / Fishing / Forestry",
    "47": "Construction / Extraction",
    "49": "Installation / Repair",
    "51": "Production",
    "53": "Transportation / Material Moving",
}

MONTH_ORDER = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}


def release_priority(row):
    """
    Higher number = later release.
    Used only to choose one duplicate row per year + major_code.
    """
    pieces = []
    for col in ["release", "source_period", "source_folder", "source_file", "source_path"]:
        if col in row.index:
            pieces.append(str(row[col]).lower())

    text = " ".join(pieces)

    best = 0
    for key, val in MONTH_ORDER.items():
        if key in text:
            best = max(best, val)

    # If release has a numeric value, use it too.
    nums = re.findall(r"\d+", text)
    if nums:
        try:
            best = max(best, max(int(n) for n in nums if len(n) <= 4))
        except Exception:
            pass

    return best


# ============================================================
# CHECK INPUT COLUMNS
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

header_cols = pd.read_csv(INPUT_FILE, nrows=0).columns.tolist()
header_lower = {c.lower(): c for c in header_cols}

required_lower = [
    "year",
    "occ_code",
    "occ_title",
    "tot_emp",
    "a_median",
    "area",
    "area_title",
    "area_type",
    "naics",
    "naics_title",
    "own_code",
]

missing_required = [c for c in required_lower if c not in header_lower]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

optional_lower = [
    "release",
    "source_workbook",
    "source_folder",
    "source_sheet",
    "source_period",
    "source_file",
    "source_path",
    "o_group",
]

usecols_lower = required_lower + [c for c in optional_lower if c in header_lower]
usecols_real = [header_lower[c] for c in usecols_lower]

print("=" * 100)
print("INPUT")
print("=" * 100)
print(INPUT_FILE)
print()
print("=" * 100)
print("OUTPUT FOLDERS")
print("=" * 100)
print(CHART_DIR)
print(TABLE_DIR)
print()
print("=" * 100)
print("COLUMNS USED")
print("=" * 100)
for c in usecols_real:
    print("-", c)
print()


# ============================================================
# READ CSV IN CHUNKS
# Keep only useful rows in memory
# ============================================================

start_time = time.time()
kept_chunks = []
total_rows_seen = 0
total_rows_kept = 0

print("=" * 100)
print("READING FILE WITH MEMORY OPTIMIZATION")
print("=" * 100)

reader = pd.read_csv(
    INPUT_FILE,
    usecols=usecols_real,
    chunksize=CHUNKSIZE,
    dtype=str,
    low_memory=False
)

for chunk_i, chunk in enumerate(reader, start=1):
    chunk_start = time.time()

    # Normalize column names to lower-case
    chunk.columns = [c.lower() for c in chunk.columns]

    total_rows_seen += len(chunk)

    # Convert year
    chunk["year"] = clean_num(chunk["year"]).astype("Int64")
    chunk = chunk[(chunk["year"] >= START_YEAR) & (chunk["year"] <= END_YEAR)]

    if chunk.empty:
        continue

    # Clean important text columns
    occ_code_txt = chunk["occ_code"].astype(str).str.strip()
    area_title_txt = normalize_text(chunk["area_title"])
    area_txt = chunk["area"].astype(str).str.strip()
    area_type_num = clean_num(chunk["area_type"])

    # U.S.-only filter
    us_mask = (
        area_title_txt.isin(["u.s.", "us", "united states"])
        | ((area_type_num == 1) & area_title_txt.str.contains(r"\bu\.?s\.?\b|united states", regex=True, na=False))
        | area_txt.isin(["99", "99.0", "0000000", "0"])
    )

    # Major occupation group only, example: 11-0000, 13-0000, 25-0000
    major_mask = occ_code_txt.str.match(r"^\d{2}-0000$", na=False)
    not_total_mask = occ_code_txt.ne("00-0000")

    chunk = chunk[us_mask & major_mask & not_total_mask].copy()

    if chunk.empty:
        continue

    chunk["_row_order"] = np.arange(total_rows_seen - len(chunk), total_rows_seen - len(chunk) + len(chunk))
    chunk["major_code"] = chunk["occ_code"].astype(str).str[:2]
    chunk["total_employment"] = clean_num(chunk["tot_emp"])
    chunk["annual_median_wage"] = clean_num(chunk["a_median"])

    kept_chunks.append(chunk)
    total_rows_kept += len(chunk)

    elapsed_min = (time.time() - start_time) / 60
    chunk_sec = time.time() - chunk_start

    print(
        f"Chunk {chunk_i:,} done | "
        f"rows seen: {total_rows_seen:,} | "
        f"rows kept: {total_rows_kept:,} | "
        f"chunk sec: {chunk_sec:.1f} | "
        f"total min: {elapsed_min:.1f}",
        flush=True
    )

    del chunk
    gc.collect()

if not kept_chunks:
    raise ValueError("No rows kept. Check U.S.-only filter, occ_code format, or input file.")

raw = pd.concat(kept_chunks, ignore_index=True)
del kept_chunks
gc.collect()

print()
print("=" * 100)
print("ROWS AFTER FIRST FILTER")
print("=" * 100)
print(f"Total rows seen: {total_rows_seen:,}")
print(f"Rows kept after U.S. + major occupation filter: {len(raw):,}")


# ============================================================
# PRIORITY FLAGS FOR DEDUPLICATION
# ============================================================

raw["naics_norm"] = raw["naics"].astype(str).str.replace(".0", "", regex=False).str.zfill(6)
raw["naics_title_norm"] = normalize_text(raw["naics_title"])
raw["own_code_norm"] = raw["own_code"].astype(str).str.replace(".0", "", regex=False).str.strip()

# Prefer national cross-industry row
raw["_all_industry_priority"] = (
    raw["naics_norm"].isin(["000000", "0000000"])
    | raw["naics_title_norm"].str.contains("cross-industry|cross industry|all industries|all industry", regex=True, na=False)
).astype(int)

# Prefer all ownership row, common OEWS all-ownership code is 1235
raw["_all_ownership_priority"] = raw["own_code_norm"].isin(["1235", "0", "00", "0000", "nan", ""]).astype(int)

# Prefer rows with real employment and wage
raw["_valid_emp_priority"] = raw["total_employment"].notna().astype(int)
raw["_valid_wage_priority"] = raw["annual_median_wage"].notna().astype(int)

# Prefer later release if source fields exist
raw["_release_priority"] = raw.apply(release_priority, axis=1)

# ============================================================
# DUPLICATE CHECK BEFORE FIX
# ============================================================

dup_before = (
    raw.groupby(["year", "major_code"], as_index=False)
    .agg(
        row_count=("occ_code", "size"),
        employment_sum=("total_employment", "sum"),
        max_employment=("total_employment", "max"),
        all_industry_rows=("_all_industry_priority", "sum"),
        all_ownership_rows=("_all_ownership_priority", "sum")
    )
)

dup_problem = dup_before[dup_before["row_count"] > 1].copy()

print()
print("=" * 100)
print("DUPLICATE CHECK BEFORE FIX")
print("=" * 100)
if dup_problem.empty:
    print("No duplicates before fix.")
else:
    print(dup_problem.sort_values(["year", "major_code"]).to_string(index=False))


# ============================================================
# FIX DUPLICATES
# One row per year + major_code
# Priority:
# 1. all-industry row
# 2. all-ownership row
# 3. valid employment/wage
# 4. later release/source
# 5. latest row order
# ============================================================

sort_cols = [
    "year",
    "major_code",
    "_all_industry_priority",
    "_all_ownership_priority",
    "_valid_emp_priority",
    "_valid_wage_priority",
    "_release_priority",
    "_row_order",
]

fixed = (
    raw.sort_values(sort_cols)
    .drop_duplicates(["year", "major_code"], keep="last")
    .copy()
)

# Keep clean columns only
keep_final_cols = [
    "year",
    "major_code",
    "occ_code",
    "occ_title",
    "total_employment",
    "annual_median_wage",
    "area",
    "area_title",
    "area_type",
    "naics",
    "naics_title",
    "own_code",
]

for c in ["release", "source_period", "source_file", "source_folder", "source_path"]:
    if c in fixed.columns:
        keep_final_cols.append(c)

fixed = fixed[keep_final_cols].copy()

dup_after = (
    fixed.groupby(["year", "major_code"], as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
)

print()
print("=" * 100)
print("DUPLICATE CHECK AFTER FIX")
print("=" * 100)
bad_after = dup_after[dup_after["row_count"] > 1]
if bad_after.empty:
    print("PASS: one row per year + major_code.")
else:
    print("WARNING: duplicates still exist.")
    print(bad_after.to_string(index=False))


# ============================================================
# TOP 12 BY FINAL YEAR EMPLOYMENT
# ============================================================

years_found = sorted(fixed["year"].dropna().astype(int).unique().tolist())
missing_years = [y for y in range(START_YEAR, END_YEAR + 1) if y not in years_found]

if END_YEAR in years_found:
    final_year = END_YEAR
else:
    final_year = max(years_found)

top_final = (
    fixed[fixed["year"] == final_year]
    .dropna(subset=["total_employment"])
    .sort_values("total_employment", ascending=False)
    .head(TOP_N)
    .copy()
)

if top_final.empty:
    raise ValueError("No top groups found for final year.")

top_codes = top_final["major_code"].astype(str).tolist()

# Canonical title = final year title
code_to_title = dict(zip(top_final["major_code"].astype(str), top_final["occ_title"].astype(str)))
code_to_short = {code: SHORT_NAMES.get(code, code_to_title.get(code, code)) for code in top_codes}

fixed["major_code"] = fixed["major_code"].astype(str)
fixed["occupation_group"] = fixed["major_code"].map(code_to_title).fillna(fixed["occ_title"])
fixed["chart_label"] = fixed["major_code"].map(code_to_short).fillna(fixed["occupation_group"])

top_data = fixed[fixed["major_code"].isin(top_codes)].copy()
top_data["occupation_group"] = top_data["major_code"].map(code_to_title)
top_data["chart_label"] = top_data["major_code"].map(code_to_short)

top_final = top_data[top_data["year"] == final_year].copy()
top_final = top_final.sort_values("total_employment", ascending=False).reset_index(drop=True)
top_final.insert(0, "rank", np.arange(1, len(top_final) + 1))

print()
print("=" * 100)
print(f"TOP {TOP_N} OCCUPATION GROUPS BY {final_year} EMPLOYMENT")
print("=" * 100)
print(
    top_final[
        ["rank", "major_code", "occupation_group", "total_employment", "annual_median_wage"]
    ].to_string(index=False)
)

print()
print("=" * 100)
print("YEAR CHECK AFTER FIX")
print("=" * 100)
print("Years found:", years_found)
print("Missing years:", missing_years)


# ============================================================
# MAKE TABLES
# ============================================================

table1 = top_final[
    ["rank", "major_code", "occupation_group", "total_employment", "annual_median_wage"]
].copy()

save_table(1, f"Top_{TOP_N}_Occupation_Groups_{final_year}_Employment", table1)

table2 = (
    top_data.pivot_table(
        index="year",
        columns="occupation_group",
        values="total_employment",
        aggfunc="first"
    )
    .reset_index()
)

save_table(2, f"Employment_Trend_Top_{TOP_N}_{START_YEAR}_{final_year}", table2)

base_year = START_YEAR if START_YEAR in years_found else min(years_found)

base_emp = top_data[top_data["year"] == base_year][["major_code", "total_employment"]].rename(
    columns={"total_employment": f"employment_{base_year}"}
)

final_emp = top_data[top_data["year"] == final_year][["major_code", "occupation_group", "total_employment"]].rename(
    columns={"total_employment": f"employment_{final_year}"}
)

table3 = final_emp.merge(base_emp, on="major_code", how="left")
table3["employment_change"] = table3[f"employment_{final_year}"] - table3[f"employment_{base_year}"]
table3["employment_percent_change"] = (
    table3["employment_change"] / table3[f"employment_{base_year}"] * 100
)
table3 = table3.sort_values("employment_change", ascending=False).reset_index(drop=True)
table3.insert(0, "rank", np.arange(1, len(table3) + 1))

save_table(3, f"Employment_Change_{base_year}_to_{final_year}", table3)

table4 = top_final[
    ["rank", "major_code", "occupation_group", "annual_median_wage", "total_employment"]
].copy().sort_values("annual_median_wage", ascending=False)

table4["rank_by_wage"] = np.arange(1, len(table4) + 1)
table4 = table4[
    ["rank_by_wage", "major_code", "occupation_group", "annual_median_wage", "total_employment"]
]

save_table(4, f"Annual_Median_Wage_Top_{TOP_N}_{final_year}", table4)

table5 = (
    top_data.pivot_table(
        index="year",
        columns="occupation_group",
        values="annual_median_wage",
        aggfunc="first"
    )
    .reset_index()
)

save_table(5, f"Annual_Median_Wage_Trend_Top_{TOP_N}_{START_YEAR}_{final_year}", table5)

base_wage = top_data[top_data["year"] == base_year][["major_code", "annual_median_wage"]].rename(
    columns={"annual_median_wage": f"wage_{base_year}"}
)

final_wage = top_data[top_data["year"] == final_year][["major_code", "occupation_group", "annual_median_wage"]].rename(
    columns={"annual_median_wage": f"wage_{final_year}"}
)

table6 = final_wage.merge(base_wage, on="major_code", how="left")
table6["wage_change"] = table6[f"wage_{final_year}"] - table6[f"wage_{base_year}"]
table6["wage_percent_change"] = (
    table6["wage_change"] / table6[f"wage_{base_year}"] * 100
)
table6 = table6.sort_values("wage_change", ascending=False).reset_index(drop=True)
table6.insert(0, "rank", np.arange(1, len(table6) + 1))

save_table(6, f"Annual_Median_Wage_Change_{base_year}_to_{final_year}", table6)

check_table = pd.DataFrame({
    "check_name": [
        "total_rows_seen",
        "rows_kept_after_us_major_filter",
        "rows_after_dedup_fix",
        "start_year",
        "final_year_used",
        "missing_years_after_fix",
        "duplicates_after_fix"
    ],
    "result": [
        total_rows_seen,
        len(raw),
        len(fixed),
        START_YEAR,
        final_year,
        ", ".join(map(str, missing_years)) if missing_years else "None",
        "PASS" if bad_after.empty else "WARNING"
    ]
})

save_table(7, "Data_Checks_After_Fix", check_table)

save_table(8, "Duplicate_Check_Before_Fix", dup_before)
save_table(9, "Duplicate_Check_After_Fix", dup_after)


# ============================================================
# MAKE CHARTS
# ============================================================

plt.rcParams.update({
    "figure.figsize": (13, 7),
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
})

# ------------------------------------------------------------
# CHART 1
# Top 12 occupation groups by final year employment
# ------------------------------------------------------------

plot1 = top_final.sort_values("total_employment", ascending=True).copy()

fig, ax = plt.subplots(figsize=(13, 7))
ax.barh(plot1["chart_label"], plot1["total_employment"])
ax.set_title(f"Top {TOP_N} U.S. Occupation Groups by Employment, {final_year}")
ax.set_xlabel("Total employment")
ax.xaxis.set_major_formatter(FuncFormatter(fmt_millions))

for i, val in enumerate(plot1["total_employment"]):
    ax.text(val, i, f" {val / 1_000_000:.1f}M", va="center", fontsize=9)

ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
save_chart(1, f"Top_{TOP_N}_Occupation_Groups_{final_year}_Employment")


# ------------------------------------------------------------
# CHART 2
# Employment trend
# ------------------------------------------------------------

pivot_emp = (
    top_data.pivot_table(
        index="year",
        columns="major_code",
        values="total_employment",
        aggfunc="first"
    )
    .reindex(range(START_YEAR, final_year + 1))
)

fig, ax = plt.subplots(figsize=(14, 8))

for code in top_codes:
    if code in pivot_emp.columns:
        ax.plot(
            pivot_emp.index,
            pivot_emp[code],
            marker="o",
            linewidth=2,
            label=code_to_short.get(code, code)
        )

ax.set_title(f"Employment Trend for Top {TOP_N} U.S. Occupation Groups, {START_YEAR}-{final_year}")
ax.set_xlabel("Year")
ax.set_ylabel("Total employment")
ax.yaxis.set_major_formatter(FuncFormatter(fmt_millions))
ax.grid(alpha=0.3)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)

plt.tight_layout()
save_chart(2, f"Employment_Trend_Top_{TOP_N}_{START_YEAR}_{final_year}")


# ------------------------------------------------------------
# CHART 3
# Employment change from base year to final year
# ------------------------------------------------------------

plot3 = table3.sort_values("employment_change", ascending=True).copy()
plot3["chart_label"] = plot3["major_code"].map(code_to_short)

fig, ax = plt.subplots(figsize=(13, 7))
ax.barh(plot3["chart_label"], plot3["employment_change"])
ax.set_title(f"Employment Change by Occupation Group, {base_year} to {final_year}")
ax.set_xlabel("Employment change")
ax.xaxis.set_major_formatter(FuncFormatter(fmt_millions))
ax.axvline(0, linewidth=1)

for i, val in enumerate(plot3["employment_change"]):
    label = f" {val / 1_000_000:+.1f}M"
    ax.text(val, i, label, va="center", fontsize=9)

ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
save_chart(3, f"Employment_Change_{base_year}_to_{final_year}")


# ------------------------------------------------------------
# CHART 4
# Annual median wage by final year
# ------------------------------------------------------------

plot4 = table4.copy()
plot4["chart_label"] = plot4["major_code"].map(code_to_short)
plot4 = plot4.sort_values("annual_median_wage", ascending=True)

fig, ax = plt.subplots(figsize=(13, 7))
ax.barh(plot4["chart_label"], plot4["annual_median_wage"])
ax.set_title(f"Annual Median Wage for Top {TOP_N} U.S. Occupation Groups, {final_year}")
ax.set_xlabel("Annual median wage")
ax.xaxis.set_major_formatter(FuncFormatter(fmt_dollar_k))

for i, val in enumerate(plot4["annual_median_wage"]):
    ax.text(val, i, f" ${val:,.0f}", va="center", fontsize=9)

ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
save_chart(4, f"Annual_Median_Wage_Top_{TOP_N}_{final_year}")


# ------------------------------------------------------------
# CHART 5
# Annual median wage trend
# ------------------------------------------------------------

pivot_wage = (
    top_data.pivot_table(
        index="year",
        columns="major_code",
        values="annual_median_wage",
        aggfunc="first"
    )
    .reindex(range(START_YEAR, final_year + 1))
)

fig, ax = plt.subplots(figsize=(14, 8))

for code in top_codes:
    if code in pivot_wage.columns:
        ax.plot(
            pivot_wage.index,
            pivot_wage[code],
            marker="o",
            linewidth=2,
            label=code_to_short.get(code, code)
        )

ax.set_title(f"Annual Median Wage Trend for Top {TOP_N} U.S. Occupation Groups, {START_YEAR}-{final_year}")
ax.set_xlabel("Year")
ax.set_ylabel("Annual median wage")
ax.yaxis.set_major_formatter(FuncFormatter(fmt_dollar_k))
ax.grid(alpha=0.3)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)

plt.tight_layout()
save_chart(5, f"Annual_Median_Wage_Trend_Top_{TOP_N}_{START_YEAR}_{final_year}")


# ------------------------------------------------------------
# CHART 6
# Employment vs wage scatter
# ------------------------------------------------------------

plot6 = top_final.copy()

fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(plot6["total_employment"], plot6["annual_median_wage"], s=80)

for _, row in plot6.iterrows():
    ax.annotate(
        code_to_short.get(row["major_code"], row["major_code"]),
        (row["total_employment"], row["annual_median_wage"]),
        textcoords="offset points",
        xytext=(6, 4),
        fontsize=8
    )

ax.set_title(f"Employment vs Annual Median Wage, Top {TOP_N} U.S. Occupation Groups, {final_year}")
ax.set_xlabel("Total employment")
ax.set_ylabel("Annual median wage")
ax.xaxis.set_major_formatter(FuncFormatter(fmt_millions))
ax.yaxis.set_major_formatter(FuncFormatter(fmt_dollar_k))
ax.grid(alpha=0.3)

plt.tight_layout()
save_chart(6, f"Employment_vs_Annual_Median_Wage_Top_{TOP_N}_{final_year}")


# ============================================================
# FINAL SUMMARY
# ============================================================

runtime_min = (time.time() - start_time) / 60

print()
print("=" * 100)
print("DONE")
print("=" * 100)
print(f"Runtime minutes: {runtime_min:.2f}")
print(f"Charts saved to: {CHART_DIR}")
print(f"Tables saved to: {TABLE_DIR}")
print()
print("Important fixes applied:")
print("1. U.S.-only rows selected")
print("2. Major occupation groups only")
print("3. One row per year + major_code")
print("4. 2012 duplicate double-counting fixed")
print("5. Education title variant fixed by using major_code")
print("6. 2010 not used because it is missing after filters")
print()
print("Chart files created:")
for p in sorted(CHART_DIR.glob("*.png")):
    print("-", p.name)

print()
print("Table files created:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print("-", p.name)

# Fix code chart 2

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import re
import gc
import time
import shutil
import textwrap

# ============================================================
# JOB CHART 2 + JOB TABLE 2
# CREATE ALL MAJOR OCCUPATION GROUPS
#
# FIXED AGAIN:
# 1. Strict U.S.-only filter
# 2. Strict all-industry filter
# 3. Major occupation groups only
# 4. Keeps all 22 major occupation groups, not only top 12
# 5. Fixes 2012 duplicate problem by choosing highest national total
# 6. Creates summary charts + individual charts for every occupation group
# ============================================================

# ============================================================
# SETTINGS
# ============================================================

INPUT_FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_1.csv"

CHART_DIR = Path.home() / "Downloads" / "Job_Chart_2"
TABLE_DIR = Path.home() / "Downloads" / "Job_Table_2"

START_YEAR = 2011
END_YEAR = 2025
CHUNKSIZE = 150_000

CLEAR_OLD_OUTPUT = True

# If later 2010 is fixed in the data, change START_YEAR to 2010.

# ============================================================
# RESET OUTPUT FOLDERS
# ============================================================

if CLEAR_OLD_OUTPUT:
    if CHART_DIR.exists():
        shutil.rmtree(CHART_DIR)
    if TABLE_DIR.exists():
        shutil.rmtree(TABLE_DIR)

CHART_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_num(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "NaN": np.nan,
            "None": np.nan,
            "<NA>": np.nan,
            "null": np.nan,
        }),
        errors="coerce"
    )


def normalize_text(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )


def safe_filename(name):
    name = re.sub(r"[^\w\s\-]", "", str(name))
    name = re.sub(r"\s+", "_", name.strip())
    return name[:160]


def fmt_millions(x, pos=None):
    try:
        return f"{x / 1_000_000:.1f}M"
    except Exception:
        return ""


def fmt_dollar_k(x, pos=None):
    try:
        return f"${x / 1000:.0f}k"
    except Exception:
        return ""


def save_table(rank, title, df):
    path = TABLE_DIR / f"{rank}_{safe_filename(title)}.csv"
    df.to_csv(path, index=False)
    print(f"Saved table {rank}: {path}", flush=True)
    return path


def save_chart(rank, title):
    path = CHART_DIR / f"{rank}_{safe_filename(title)}.png"
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"Saved chart {rank}: {path}", flush=True)
    return path


def wrap_label(text, width=24):
    return "\n".join(textwrap.wrap(str(text), width=width))


SHORT_NAMES = {
    "11": "Management",
    "13": "Business / Financial",
    "15": "Computer / Math",
    "17": "Architecture / Engineering",
    "19": "Science",
    "21": "Community / Social Service",
    "23": "Legal",
    "25": "Education / Library",
    "27": "Arts / Media",
    "29": "Healthcare Practitioners",
    "31": "Healthcare Support",
    "33": "Protective Service",
    "35": "Food Serving",
    "37": "Building / Grounds",
    "39": "Personal Care / Service",
    "41": "Sales",
    "43": "Office / Admin Support",
    "45": "Farming / Fishing / Forestry",
    "47": "Construction / Extraction",
    "49": "Installation / Repair",
    "51": "Production",
    "53": "Transportation / Material Moving",
}

# ============================================================
# CHECK INPUT
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

header_cols = pd.read_csv(INPUT_FILE, nrows=0).columns.tolist()
header_lower = {c.lower(): c for c in header_cols}

required_lower = [
    "year",
    "occ_code",
    "occ_title",
    "tot_emp",
    "a_median",
    "area",
    "area_title",
    "area_type",
    "naics",
    "naics_title",
    "own_code",
]

missing_required = [c for c in required_lower if c not in header_lower]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

optional_lower = [
    "release",
    "source_workbook",
    "source_folder",
    "source_sheet",
    "source_period",
    "source_file",
    "source_path",
    "o_group",
]

usecols_lower = required_lower + [c for c in optional_lower if c in header_lower]
usecols_real = [header_lower[c] for c in usecols_lower]

print("=" * 100)
print("INPUT FILE")
print("=" * 100)
print(INPUT_FILE)

print()
print("=" * 100)
print("OUTPUT FOLDERS")
print("=" * 100)
print(CHART_DIR)
print(TABLE_DIR)

print()
print("=" * 100)
print("COLUMNS USED")
print("=" * 100)
for c in usecols_real:
    print("-", c)

# ============================================================
# READ AND STRICT FILTER
# ============================================================

start_time = time.time()

kept_chunks = []
total_rows_seen = 0
total_after_year = 0
total_after_major = 0
total_after_us = 0
total_after_all_industry = 0

print()
print("=" * 100)
print("READING FILE WITH MEMORY OPTIMIZATION")
print("=" * 100)

reader = pd.read_csv(
    INPUT_FILE,
    usecols=usecols_real,
    chunksize=CHUNKSIZE,
    dtype=str,
    low_memory=False
)

for chunk_i, chunk in enumerate(reader, start=1):
    chunk_start = time.time()

    chunk.columns = [c.lower() for c in chunk.columns]
    total_rows_seen += len(chunk)

    chunk["year"] = clean_num(chunk["year"]).astype("Int64")
    chunk = chunk[(chunk["year"] >= START_YEAR) & (chunk["year"] <= END_YEAR)].copy()
    total_after_year += len(chunk)

    if chunk.empty:
        continue

    # -------------------------------
    # Major occupation groups only
    # example: 11-0000, 13-0000
    # exclude 00-0000 all occupations
    # -------------------------------
    occ_code_txt = chunk["occ_code"].astype(str).str.strip()
    major_mask = occ_code_txt.str.match(r"^\d{2}-0000$", na=False)
    not_total_mask = occ_code_txt.ne("00-0000")

    chunk = chunk[major_mask & not_total_mask].copy()
    total_after_major += len(chunk)

    if chunk.empty:
        continue

    # -------------------------------
    # Strict U.S. only
    # Do NOT use area == 0 alone.
    # That was the problem before.
    # -------------------------------
    area_title_norm = normalize_text(chunk["area_title"])
    area_type_num = clean_num(chunk["area_type"])
    area_code_norm = (
        chunk["area"]
        .astype(str)
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.zfill(2)
    )

    us_mask = (
        area_title_norm.isin(["u.s.", "us", "united states"])
        | ((area_type_num == 1) & area_code_norm.isin(["99"]))
    )

    chunk = chunk[us_mask].copy()
    total_after_us += len(chunk)

    if chunk.empty:
        continue

    # -------------------------------
    # Strict all-industry / cross-industry only
    # This removes state/industry duplicate rows.
    # -------------------------------
    naics_norm = (
        chunk["naics"]
        .astype(str)
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.replace("-", "", regex=False)
        .str.zfill(6)
    )

    naics_title_norm = normalize_text(chunk["naics_title"])

    all_industry_mask = (
        naics_norm.isin(["000000"])
        | naics_title_norm.str.contains(
            "cross-industry|cross industry|all industries|all industry",
            regex=True,
            na=False
        )
    )

    chunk = chunk[all_industry_mask].copy()
    total_after_all_industry += len(chunk)

    if chunk.empty:
        continue

    chunk["major_code"] = chunk["occ_code"].astype(str).str[:2]
    chunk["total_employment"] = clean_num(chunk["tot_emp"])
    chunk["annual_median_wage"] = clean_num(chunk["a_median"])

    # Save row order for debugging
    chunk["_source_row_order"] = np.arange(len(chunk))

    kept_chunks.append(chunk)

    elapsed_min = (time.time() - start_time) / 60
    chunk_sec = time.time() - chunk_start

    print(
        f"Chunk {chunk_i:,} done | "
        f"rows seen: {total_rows_seen:,} | "
        f"kept after strict filters: {total_after_all_industry:,} | "
        f"chunk sec: {chunk_sec:.1f} | "
        f"total min: {elapsed_min:.1f}",
        flush=True
    )

    del chunk
    gc.collect()

if not kept_chunks:
    raise ValueError("No rows kept. Check area_title, area_type, naics, and occ_code filters.")

raw = pd.concat(kept_chunks, ignore_index=True)
del kept_chunks
gc.collect()

print()
print("=" * 100)
print("ROW CHECK BEFORE DEDUP")
print("=" * 100)
print(f"Total rows seen:                  {total_rows_seen:,}")
print(f"Rows after year filter:           {total_after_year:,}")
print(f"Rows after major group filter:    {total_after_major:,}")
print(f"Rows after strict U.S. filter:    {total_after_us:,}")
print(f"Rows after all-industry filter:   {total_after_all_industry:,}")
print(f"Rows in raw filtered data:        {len(raw):,}")

# ============================================================
# DUPLICATE CHECK BEFORE FIX
# ============================================================

dup_before = (
    raw.groupby(["year", "major_code"], as_index=False)
    .agg(
        row_count=("occ_code", "size"),
        employment_sum=("total_employment", "sum"),
        max_employment=("total_employment", "max"),
        min_employment=("total_employment", "min"),
        max_wage=("annual_median_wage", "max"),
        min_wage=("annual_median_wage", "min")
    )
)

dup_problem = dup_before[dup_before["row_count"] > 1].copy()

print()
print("=" * 100)
print("DUPLICATE CHECK BEFORE FIX")
print("=" * 100)
if dup_problem.empty:
    print("No duplicates before fix.")
else:
    print(dup_problem.sort_values(["year", "major_code"]).to_string(index=False))

# ============================================================
# FIX DUPLICATES
#
# Important:
# For 2012, duplicate rows can include a lower wrong total.
# For national all-industry all-ownership trend, choose the row
# with the highest total employment for year + major_code.
# ============================================================

raw["_valid_emp"] = raw["total_employment"].notna().astype(int)
raw["_valid_wage"] = raw["annual_median_wage"].notna().astype(int)

own_norm = (
    raw["own_code"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(".0", "", regex=False)
)

raw["_own_priority"] = own_norm.isin(["1235", "nan", "none", "<na>", ""]).astype(int)

fixed = (
    raw.sort_values(
        [
            "year",
            "major_code",
            "_valid_emp",
            "_own_priority",
            "total_employment",
            "_valid_wage",
            "annual_median_wage",
        ],
        ascending=[True, True, True, True, True, True, True]
    )
    .drop_duplicates(["year", "major_code"], keep="last")
    .copy()
)

# Canonical occupation title:
# use latest available title for each major_code
title_map = (
    fixed.sort_values("year")
    .groupby("major_code")["occ_title"]
    .last()
    .to_dict()
)

fixed["occupation_group"] = fixed["major_code"].map(title_map)
fixed["chart_label"] = fixed["major_code"].map(SHORT_NAMES).fillna(fixed["occupation_group"])

final_cols = [
    "year",
    "major_code",
    "occ_code",
    "occupation_group",
    "chart_label",
    "total_employment",
    "annual_median_wage",
    "area",
    "area_title",
    "area_type",
    "naics",
    "naics_title",
    "own_code",
]

for c in ["release", "source_period", "source_file", "source_folder", "source_path", "o_group"]:
    if c in fixed.columns:
        final_cols.append(c)

fixed = fixed[final_cols].copy()
fixed = fixed.sort_values(["year", "major_code"]).reset_index(drop=True)

# ============================================================
# CHECK AFTER FIX
# ============================================================

dup_after = (
    fixed.groupby(["year", "major_code"], as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
)

bad_after = dup_after[dup_after["row_count"] > 1]

years_found = sorted(fixed["year"].dropna().astype(int).unique().tolist())
missing_years = [y for y in range(START_YEAR, END_YEAR + 1) if y not in years_found]

major_codes_found = sorted(fixed["major_code"].astype(str).unique().tolist())
final_year = END_YEAR if END_YEAR in years_found else max(years_found)
base_year = START_YEAR if START_YEAR in years_found else min(years_found)

print()
print("=" * 100)
print("CHECK AFTER FIX")
print("=" * 100)
print(f"Rows after dedup fix: {len(fixed):,}")
print(f"Years found: {years_found}")
print(f"Missing years: {missing_years if missing_years else 'None'}")
print(f"Major occupation groups found: {len(major_codes_found)}")
print(f"Duplicate after fix: {'PASS' if bad_after.empty else 'WARNING'}")

if not bad_after.empty:
    print(bad_after.to_string(index=False))

# ============================================================
# CREATE TABLES
# ============================================================

final_summary = (
    fixed[fixed["year"] == final_year]
    .sort_values("total_employment", ascending=False)
    .reset_index(drop=True)
)

final_summary.insert(0, "rank_by_employment", np.arange(1, len(final_summary) + 1))

save_table(
    1,
    f"All_Major_Occupation_Groups_{final_year}_Summary",
    final_summary[
        [
            "rank_by_employment",
            "major_code",
            "occupation_group",
            "total_employment",
            "annual_median_wage",
        ]
    ]
)

save_table(
    2,
    f"All_Major_Occupation_Groups_Long_Fixed_{base_year}_{final_year}",
    fixed
)

employment_pivot = (
    fixed.pivot_table(
        index="year",
        columns="occupation_group",
        values="total_employment",
        aggfunc="first"
    )
    .reset_index()
)

save_table(
    3,
    f"All_Major_Occupation_Groups_Employment_Pivot_{base_year}_{final_year}",
    employment_pivot
)

wage_pivot = (
    fixed.pivot_table(
        index="year",
        columns="occupation_group",
        values="annual_median_wage",
        aggfunc="first"
    )
    .reset_index()
)

save_table(
    4,
    f"All_Major_Occupation_Groups_Wage_Pivot_{base_year}_{final_year}",
    wage_pivot
)

base_emp = fixed[fixed["year"] == base_year][["major_code", "total_employment", "annual_median_wage"]].rename(
    columns={
        "total_employment": f"employment_{base_year}",
        "annual_median_wage": f"wage_{base_year}",
    }
)

final_emp = fixed[fixed["year"] == final_year][
    ["major_code", "occupation_group", "chart_label", "total_employment", "annual_median_wage"]
].rename(
    columns={
        "total_employment": f"employment_{final_year}",
        "annual_median_wage": f"wage_{final_year}",
    }
)

change_table = final_emp.merge(base_emp, on="major_code", how="left")

change_table["employment_change"] = (
    change_table[f"employment_{final_year}"] - change_table[f"employment_{base_year}"]
)

change_table["employment_percent_change"] = (
    change_table["employment_change"] / change_table[f"employment_{base_year}"] * 100
)

change_table["wage_change"] = (
    change_table[f"wage_{final_year}"] - change_table[f"wage_{base_year}"]
)

change_table["wage_percent_change"] = (
    change_table["wage_change"] / change_table[f"wage_{base_year}"] * 100
)

change_table = change_table.sort_values("employment_change", ascending=False).reset_index(drop=True)
change_table.insert(0, "rank_by_employment_change", np.arange(1, len(change_table) + 1))

save_table(
    5,
    f"All_Major_Occupation_Groups_Change_{base_year}_to_{final_year}",
    change_table
)

check_table = pd.DataFrame({
    "check_name": [
        "total_rows_seen",
        "rows_after_year_filter",
        "rows_after_major_group_filter",
        "rows_after_strict_us_filter",
        "rows_after_all_industry_filter",
        "rows_after_dedup_fix",
        "base_year",
        "final_year",
        "missing_years",
        "major_groups_found",
        "duplicate_after_fix",
    ],
    "result": [
        total_rows_seen,
        total_after_year,
        total_after_major,
        total_after_us,
        total_after_all_industry,
        len(fixed),
        base_year,
        final_year,
        ", ".join(map(str, missing_years)) if missing_years else "None",
        len(major_codes_found),
        "PASS" if bad_after.empty else "WARNING",
    ]
})

save_table(6, "Data_Checks_After_Strict_Fix", check_table)
save_table(7, "Duplicate_Check_Before_Strict_Fix", dup_before)
save_table(8, "Duplicate_Check_After_Strict_Fix", dup_after)

# ============================================================
# CHART SETTINGS
# ============================================================

plt.rcParams.update({
    "figure.figsize": (14, 8),
    "axes.titlesize": 17,
    "axes.labelsize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
})

chart_rank = 1

# ============================================================
# CHART 1
# ALL groups by employment final year
# ============================================================

plot = final_summary.sort_values("total_employment", ascending=True).copy()
plot["plot_label"] = plot["chart_label"].apply(lambda x: wrap_label(x, 26))

fig, ax = plt.subplots(figsize=(14, 10))
ax.barh(plot["plot_label"], plot["total_employment"])
ax.set_title(f"All U.S. Major Occupation Groups by Employment, {final_year}")
ax.set_xlabel("Total employment")
ax.xaxis.set_major_formatter(FuncFormatter(fmt_millions))
ax.grid(axis="x", alpha=0.3)

for i, val in enumerate(plot["total_employment"]):
    ax.text(val, i, f" {val / 1_000_000:.1f}M", va="center", fontsize=8)

plt.tight_layout()
save_chart(chart_rank, f"All_Major_Occupation_Groups_{final_year}_Employment")
chart_rank += 1

# ============================================================
# CHART 2
# ALL groups by employment change
# ============================================================

plot = change_table.sort_values("employment_change", ascending=True).copy()
plot["plot_label"] = plot["chart_label"].apply(lambda x: wrap_label(x, 26))

fig, ax = plt.subplots(figsize=(14, 10))
ax.barh(plot["plot_label"], plot["employment_change"])
ax.set_title(f"Employment Change for All U.S. Major Occupation Groups, {base_year} to {final_year}")
ax.set_xlabel("Employment change")
ax.xaxis.set_major_formatter(FuncFormatter(fmt_millions))
ax.axvline(0, linewidth=1)
ax.grid(axis="x", alpha=0.3)

for i, val in enumerate(plot["employment_change"]):
    ax.text(val, i, f" {val / 1_000_000:+.1f}M", va="center", fontsize=8)

plt.tight_layout()
save_chart(chart_rank, f"All_Major_Occupation_Groups_Employment_Change_{base_year}_to_{final_year}")
chart_rank += 1

# ============================================================
# CHART 3
# ALL groups by annual median wage final year
# ============================================================

plot = final_summary.sort_values("annual_median_wage", ascending=True).copy()
plot["plot_label"] = plot["chart_label"].apply(lambda x: wrap_label(x, 26))

fig, ax = plt.subplots(figsize=(14, 10))
ax.barh(plot["plot_label"], plot["annual_median_wage"])
ax.set_title(f"Annual Median Wage for All U.S. Major Occupation Groups, {final_year}")
ax.set_xlabel("Annual median wage")
ax.xaxis.set_major_formatter(FuncFormatter(fmt_dollar_k))
ax.grid(axis="x", alpha=0.3)

for i, val in enumerate(plot["annual_median_wage"]):
    ax.text(val, i, f" ${val:,.0f}", va="center", fontsize=8)

plt.tight_layout()
save_chart(chart_rank, f"All_Major_Occupation_Groups_{final_year}_Annual_Median_Wage")
chart_rank += 1

# ============================================================
# CHART 4
# ALL groups by wage change
# ============================================================

plot = change_table.sort_values("wage_change", ascending=True).copy()
plot["plot_label"] = plot["chart_label"].apply(lambda x: wrap_label(x, 26))

fig, ax = plt.subplots(figsize=(14, 10))
ax.barh(plot["plot_label"], plot["wage_change"])
ax.set_title(f"Annual Median Wage Change for All U.S. Major Occupation Groups, {base_year} to {final_year}")
ax.set_xlabel("Annual median wage change")
ax.xaxis.set_major_formatter(FuncFormatter(fmt_dollar_k))
ax.axvline(0, linewidth=1)
ax.grid(axis="x", alpha=0.3)

for i, val in enumerate(plot["wage_change"]):
    ax.text(val, i, f" ${val:+,.0f}", va="center", fontsize=8)

plt.tight_layout()
save_chart(chart_rank, f"All_Major_Occupation_Groups_Wage_Change_{base_year}_to_{final_year}")
chart_rank += 1

# ============================================================
# CHART 5
# Top 12 employment trend
# ============================================================

top_codes = final_summary.head(12)["major_code"].astype(str).tolist()

pivot_emp = (
    fixed[fixed["major_code"].isin(top_codes)]
    .pivot_table(index="year", columns="major_code", values="total_employment", aggfunc="first")
    .reindex(range(base_year, final_year + 1))
)

fig, ax = plt.subplots(figsize=(15, 8))

for code in top_codes:
    label = SHORT_NAMES.get(code, title_map.get(code, code))
    ax.plot(pivot_emp.index, pivot_emp[code], marker="o", linewidth=2, label=label)

ax.set_title(f"Employment Trend for Top 12 U.S. Occupation Groups, {base_year}-{final_year}")
ax.set_xlabel("Year")
ax.set_ylabel("Total employment")
ax.yaxis.set_major_formatter(FuncFormatter(fmt_millions))
ax.grid(alpha=0.3)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)

plt.tight_layout()
save_chart(chart_rank, f"Top_12_Employment_Trend_{base_year}_{final_year}")
chart_rank += 1

# ============================================================
# CHART 6
# Top 12 wage trend
# ============================================================

pivot_wage = (
    fixed[fixed["major_code"].isin(top_codes)]
    .pivot_table(index="year", columns="major_code", values="annual_median_wage", aggfunc="first")
    .reindex(range(base_year, final_year + 1))
)

fig, ax = plt.subplots(figsize=(15, 8))

for code in top_codes:
    label = SHORT_NAMES.get(code, title_map.get(code, code))
    ax.plot(pivot_wage.index, pivot_wage[code], marker="o", linewidth=2, label=label)

ax.set_title(f"Annual Median Wage Trend for Top 12 U.S. Occupation Groups, {base_year}-{final_year}")
ax.set_xlabel("Year")
ax.set_ylabel("Annual median wage")
ax.yaxis.set_major_formatter(FuncFormatter(fmt_dollar_k))
ax.grid(alpha=0.3)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)

plt.tight_layout()
save_chart(chart_rank, f"Top_12_Annual_Median_Wage_Trend_{base_year}_{final_year}")
chart_rank += 1

# ============================================================
# CHART 7
# Employment vs wage scatter, all groups
# ============================================================

plot = final_summary.copy()

fig, ax = plt.subplots(figsize=(14, 8))
ax.scatter(plot["total_employment"], plot["annual_median_wage"], s=80)

for _, row in plot.iterrows():
    ax.annotate(
        row["chart_label"],
        (row["total_employment"], row["annual_median_wage"]),
        textcoords="offset points",
        xytext=(6, 4),
        fontsize=8
    )

ax.set_title(f"Employment vs Annual Median Wage, All U.S. Major Occupation Groups, {final_year}")
ax.set_xlabel("Total employment")
ax.set_ylabel("Annual median wage")
ax.xaxis.set_major_formatter(FuncFormatter(fmt_millions))
ax.yaxis.set_major_formatter(FuncFormatter(fmt_dollar_k))
ax.grid(alpha=0.3)

plt.tight_layout()
save_chart(chart_rank, f"All_Major_Occupation_Groups_Employment_vs_Wage_{final_year}")
chart_rank += 1

# ============================================================
# INDIVIDUAL CHARTS FOR EVERY OCCUPATION GROUP
# ============================================================

individual_dir = CHART_DIR / "Individual_Occupation_Group_Charts"
individual_dir.mkdir(parents=True, exist_ok=True)

old_chart_dir = CHART_DIR
CHART_DIR = individual_dir

for code in major_codes_found:
    group_df = fixed[fixed["major_code"] == code].sort_values("year").copy()

    if group_df.empty:
        continue

    label = SHORT_NAMES.get(code, group_df["occupation_group"].iloc[-1])
    full_title = group_df["occupation_group"].iloc[-1]

    # Employment trend individual
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(group_df["year"], group_df["total_employment"], marker="o", linewidth=2)
    ax.set_title(f"Employment Trend: {full_title}")
    ax.set_xlabel("Year")
    ax.set_ylabel("Total employment")
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_millions))
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_chart(chart_rank, f"{code}_Employment_Trend_{safe_filename(label)}_{base_year}_{final_year}")
    chart_rank += 1

    # Wage trend individual
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(group_df["year"], group_df["annual_median_wage"], marker="o", linewidth=2)
    ax.set_title(f"Annual Median Wage Trend: {full_title}")
    ax.set_xlabel("Year")
    ax.set_ylabel("Annual median wage")
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_dollar_k))
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_chart(chart_rank, f"{code}_Wage_Trend_{safe_filename(label)}_{base_year}_{final_year}")
    chart_rank += 1

CHART_DIR = old_chart_dir

# ============================================================
# FINAL PRINT SUMMARY
# ============================================================

runtime_min = (time.time() - start_time) / 60

print()
print("=" * 100)
print("DONE")
print("=" * 100)
print(f"Runtime minutes: {runtime_min:.2f}")
print(f"Charts saved to: {old_chart_dir}")
print(f"Individual charts saved to: {individual_dir}")
print(f"Tables saved to: {TABLE_DIR}")
print()
print("Important fixes applied:")
print("1. Strict U.S.-only filter")
print("2. Strict all-industry filter")
print("3. Major occupation groups only")
print("4. All major groups created, not only top 12")
print("5. Duplicate rows fixed by choosing highest national total employment")
print("6. 2010 not used because it is missing after filters")
print()
print(f"Total chart PNG files created: {len(list(old_chart_dir.rglob('*.png')))}")
print(f"Total table CSV files created: {len(list(TABLE_DIR.glob('*.csv')))}")
print()
print("Main chart files:")
for p in sorted(old_chart_dir.glob("*.png")):
    print("-", p.name)

print()
print("Table files:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print("-", p.name)

# Fix code chart 3

In [ ]:
from pathlib import Path
import pandas as pd
import gc
import time

# ============================================================
# READ + FILTER + FINAL DEDUP CHECK ONLY
# MEMORY OPTIMIZED
#
# NO SAVE
# NO CHART
# NO NEW CSV
# ============================================================

DOWNLOADS = Path.home() / "Downloads"

POSSIBLE_FILES = [
    DOWNLOADS / "all_2010-2025_job_CLEANED_numeric_only_1.csv",
    DOWNLOADS / "all_2010-2025_job.csv",
]

INPUT_FILE = None
for p in POSSIBLE_FILES:
    if p.exists():
        INPUT_FILE = p
        break

if INPUT_FILE is None:
    raise FileNotFoundError("Could not find job CSV in Downloads. Check the filename.")

print("=" * 100)
print("INPUT FILE")
print("=" * 100)
print(INPUT_FILE)

# ============================================================
# SETTINGS
# ============================================================

START_YEAR = 2011
END_YEAR = 2025
CHUNKSIZE = 150_000

USE_COLS = [
    "year",
    "occ_code",
    "occ_title",
    "tot_emp",
    "a_median",
    "area",
    "area_title",
    "area_type",
    "naics",
    "naics_title",
    "own_code",
    "o_group",
]

keep_parts = []

rows_seen = 0
rows_kept = 0
start_time = time.time()

print()
print("=" * 100)
print("READING FILE WITH MEMORY OPTIMIZATION")
print("=" * 100)

for chunk_num, chunk in enumerate(
    pd.read_csv(
        INPUT_FILE,
        usecols=lambda c: c in USE_COLS,
        chunksize=CHUNKSIZE,
        low_memory=False,
    ),
    start=1
):
    rows_seen += len(chunk)

    # lowercase column names just in case
    chunk.columns = [c.lower().strip() for c in chunk.columns]

    # numeric conversion
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["tot_emp"] = pd.to_numeric(chunk["tot_emp"], errors="coerce")
    chunk["a_median"] = pd.to_numeric(chunk["a_median"], errors="coerce")

    # clean text columns
    for col in ["occ_code", "occ_title", "area", "area_title", "naics", "naics_title", "own_code", "o_group"]:
        if col in chunk.columns:
            chunk[col] = chunk[col].astype(str).str.strip()

    # create major code
    chunk["major_code"] = chunk["occ_code"].astype(str).str[:2]

    # ========================================================
    # STRICT FILTER
    # ========================================================
    mask = (
        (chunk["year"] >= START_YEAR)
        & (chunk["year"] <= END_YEAR)

        # major occupation groups only
        & (chunk["occ_code"].astype(str).str.match(r"^\d{2}-0000$", na=False))

        # U.S. only
        & (
            (chunk["area"].astype(str).isin(["99", "0000000", "0"]))
            | (chunk["area_title"].str.upper().isin(["U.S.", "US", "UNITED STATES"]))
        )

        # all industry only
        & (
            (chunk["naics"].astype(str).isin(["000000", "0"]))
            | (chunk["naics_title"].str.upper().str.contains("CROSS-INDUSTRY|ALL INDUSTRIES", na=False))
        )

        # IMPORTANT FIX:
        # own_code 1235 = all ownerships.
        # This removes private/government duplicate rows.
        & (chunk["own_code"].astype(str).isin(["1235", "1235.0"]))

        & (chunk["tot_emp"].notna())
        & (chunk["a_median"].notna())
    )

    kept = chunk.loc[mask].copy()
    rows_kept += len(kept)

    if len(kept) > 0:
        keep_parts.append(kept)

    if chunk_num == 1 or chunk_num % 5 == 0:
        elapsed_min = (time.time() - start_time) / 60
        print(
            f"Chunk {chunk_num:,} done | rows seen: {rows_seen:,} | kept: {rows_kept:,} | total min: {elapsed_min:.1f}",
            flush=True
        )

    del chunk, kept
    gc.collect()

print()
print("=" * 100)
print("READ COMPLETE")
print("=" * 100)
print(f"Rows seen: {rows_seen:,}")
print(f"Rows kept before final dedup: {rows_kept:,}")

if not keep_parts:
    raise ValueError("No rows kept. Filter is too strict or column values are different.")

raw_filtered = pd.concat(keep_parts, ignore_index=True)

# ============================================================
# FINAL DEDUP FIX
# ============================================================

final_df = raw_filtered.copy()

final_df["year"] = pd.to_numeric(final_df["year"], errors="coerce")
final_df["tot_emp"] = pd.to_numeric(final_df["tot_emp"], errors="coerce")
final_df["a_median"] = pd.to_numeric(final_df["a_median"], errors="coerce")
final_df["major_code"] = final_df["occ_code"].astype(str).str[:2].str.zfill(2)

final_df = final_df.dropna(subset=["year", "major_code", "tot_emp", "a_median"])
final_df["year"] = final_df["year"].astype(int)

# Do not sum duplicates.
# Keep one best national all-ownership row per year + major occupation group.
final_df = (
    final_df
    .sort_values(
        by=["year", "major_code", "tot_emp", "a_median"],
        ascending=[True, True, False, False]
    )
    .drop_duplicates(subset=["year", "major_code"], keep="first")
    .reset_index(drop=True)
)

# ============================================================
# CHECK AFTER FIX
# ============================================================

print()
print("=" * 100)
print("CHECK AFTER FIX")
print("=" * 100)

expected_rows = (END_YEAR - START_YEAR + 1) * 22
duplicate_count = final_df.duplicated(["year", "major_code"]).sum()

print(f"Rows after final fix:      {len(final_df):,}")
print(f"Expected rows:             {expected_rows:,}")
print(f"Duplicate year+major_code: {duplicate_count:,}")
print(f"Year range:                {final_df['year'].min()} to {final_df['year'].max()}")

groups_per_year = (
    final_df
    .groupby("year")["major_code"]
    .nunique()
    .reset_index(name="group_count")
)

print()
print("Groups per year:")
print(groups_per_year.to_string(index=False))

bad_years = groups_per_year[groups_per_year["group_count"] != 22]

print()
if len(final_df) == expected_rows and duplicate_count == 0 and bad_years.empty:
    print("✅ PASS: Final chart table is clean.")
    print("Now use final_df for all charts.")
else:
    print("❌ WARNING: Still not clean.")
    print("Bad years:")
    print(bad_years.to_string(index=False))

# Fix code chart 4

In [ ]:
from pathlib import Path
import pandas as pd
import gc

# ============================================================
# CHECK ONLY: ARE 2010 AND 2012 INSIDE CLEANED FILE?
# NO SAVE
# NO CHANGE
# MEMORY OPTIMIZED
# ============================================================

FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_1.csv"

CHUNKSIZE = 150_000

needed_cols = [
    "year",
    "area",
    "area_title",
    "naics",
    "naics_title",
    "own_code",
    "occ_code",
    "occ_title",
    "o_group",
    "i_group",
    "tot_emp"
]

print("=" * 100)
print("CHECK ONLY: 2010 AND 2012 IN CLEANED FILE")
print("=" * 100)

year_counts = {}
major_candidate_counts = {}
sample_rows = []

for chunk_num, chunk in enumerate(
    pd.read_csv(FILE, chunksize=CHUNKSIZE, low_memory=False),
    start=1
):
    # keep only columns that exist
    cols = [c for c in needed_cols if c in chunk.columns]
    chunk = chunk[cols].copy()

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk = chunk[chunk["year"].isin([2010, 2012])]

    if len(chunk) > 0:
        for y, n in chunk["year"].value_counts().items():
            year_counts[int(y)] = year_counts.get(int(y), 0) + int(n)

        # Normalize occ_code
        if "occ_code" in chunk.columns:
            occ = chunk["occ_code"].astype(str).str.strip()
            major_mask = occ.str.match(r"^\d{2}-0000$", na=False)

            for y, n in chunk.loc[major_mask, "year"].value_counts().items():
                major_candidate_counts[int(y)] = major_candidate_counts.get(int(y), 0) + int(n)

        # keep small sample only
        if len(sample_rows) < 50:
            sample_rows.append(chunk.head(10))

    if chunk_num == 1 or chunk_num % 5 == 0:
        print(f"Chunk {chunk_num} checked...", flush=True)

    del chunk
    gc.collect()

print("\n" + "=" * 100)
print("RESULT")
print("=" * 100)

print("Total rows found by year:")
for y in [2010, 2012]:
    print(f"{y}: {year_counts.get(y, 0):,}")

print("\nMajor occupation candidate rows by occ_code xx-0000:")
for y in [2010, 2012]:
    print(f"{y}: {major_candidate_counts.get(y, 0):,}")

if sample_rows:
    sample_df = pd.concat(sample_rows, ignore_index=True)
    print("\nSample rows from 2010/2012:")
    display(sample_df.head(50))
else:
    print("\nNo 2010 or 2012 rows found in the cleaned file.")

# Fix code chart 5

In [ ]:
from pathlib import Path
import pandas as pd
import gc
import time

# ============================================================
# FINAL FIX FROM CLEANED FILE
# DO NOT CREATE CLEANED _2
#
# Problem:
# 2010 and 2012 exist in _1, but were dropped by filter.
#
# Fix:
# Use occ_code xx-0000 to identify major occupation groups.
# Then keep ONE best row per year + occ_code by highest tot_emp.
#
# This avoids depending on o_group / i_group because they are NaN
# for 2010 and 2012.
# ============================================================

INPUT_FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_1.csv"

OUTPUT_FILE = Path.home() / "Downloads" / "job_major_occupation_FINAL_2010-2025_from_cleaned_1.csv"

CHUNKSIZE = 150_000

START_YEAR = 2010
END_YEAR = 2025

KEEP_COLS = [
    "year",
    "area",
    "area_title",
    "naics",
    "naics_title",
    "own_code",
    "occ_code",
    "occ_title",
    "o_group",
    "i_group",
    "tot_emp",
]

print("=" * 100)
print("READING CLEANED _1 FILE WITH MEMORY OPTIMIZATION")
print("=" * 100)

kept_parts = []
rows_seen = 0
rows_kept = 0
start_time = time.time()

for chunk_num, chunk in enumerate(
    pd.read_csv(INPUT_FILE, chunksize=CHUNKSIZE, low_memory=False),
    start=1
):
    rows_seen += len(chunk)

    # Keep only columns that exist
    cols = [c for c in KEEP_COLS if c in chunk.columns]
    chunk = chunk[cols].copy()

    # Clean year
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    # Clean employment
    chunk["tot_emp"] = (
        chunk["tot_emp"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.strip()
    )
    chunk["tot_emp"] = pd.to_numeric(chunk["tot_emp"], errors="coerce")

    # Clean occupation code
    chunk["occ_code"] = chunk["occ_code"].astype(str).str.strip()

    # Keep 2010-2025 only
    chunk = chunk[
        (chunk["year"] >= START_YEAR) &
        (chunk["year"] <= END_YEAR)
    ]

    # Keep major occupation groups only: xx-0000
    major_mask = chunk["occ_code"].str.match(r"^\d{2}-0000$", na=False)
    chunk = chunk[major_mask].copy()

    # Remove rows with missing employment
    chunk = chunk[chunk["tot_emp"].notna()].copy()

    if len(chunk) > 0:
        kept_parts.append(chunk)
        rows_kept += len(chunk)

    if chunk_num == 1 or chunk_num % 5 == 0:
        mins = (time.time() - start_time) / 60
        print(
            f"Chunk {chunk_num} done | rows seen: {rows_seen:,} | "
            f"kept candidates: {rows_kept:,} | total min: {mins:.1f}",
            flush=True
        )

    del chunk
    gc.collect()

print("\n" + "=" * 100)
print("READ COMPLETE")
print("=" * 100)
print(f"Rows seen: {rows_seen:,}")
print(f"Major occupation candidate rows kept: {rows_kept:,}")

# ============================================================
# COMBINE ONLY SMALL KEPT DATA
# ============================================================

major_df = pd.concat(kept_parts, ignore_index=True)

print("\n" + "=" * 100)
print("DEDUP FIX")
print("=" * 100)

# For each year + occupation code, keep the row with the highest employment.
# This usually selects the national / U.S. cross-industry row.
major_df = major_df.sort_values(
    ["year", "occ_code", "tot_emp"],
    ascending=[True, True, False]
)

final_df = major_df.drop_duplicates(
    subset=["year", "occ_code"],
    keep="first"
).copy()

final_df = final_df.sort_values(["year", "occ_code"]).reset_index(drop=True)

# Rename for chart clarity
final_df = final_df.rename(columns={
    "occ_code": "major_code",
    "occ_title": "major_occupation_group"
})

# ============================================================
# FINAL CHECK
# ============================================================

group_check = (
    final_df
    .groupby("year")["major_code"]
    .nunique()
    .reset_index(name="group_count")
)

duplicate_count = final_df.duplicated(["year", "major_code"]).sum()

expected_years = END_YEAR - START_YEAR + 1
expected_groups_per_year = group_check["group_count"].max()
expected_rows = expected_years * expected_groups_per_year

print("\n" + "=" * 100)
print("CHECK AFTER FINAL FIX")
print("=" * 100)

print(f"Rows after final fix:      {len(final_df):,}")
print(f"Expected years:            {expected_years}")
print(f"Expected groups per year:  {expected_groups_per_year}")
print(f"Expected rows:             {expected_rows:,}")
print(f"Duplicate year+major_code: {duplicate_count}")
print(f"Year range:                {int(final_df['year'].min())} to {int(final_df['year'].max())}")

print("\nGroups per year:")
print(group_check.to_string(index=False))

bad_years = group_check[group_check["group_count"] != expected_groups_per_year]

if duplicate_count == 0 and bad_years.empty and len(final_df) == expected_rows:
    print("\n✅ CLEAN NOW.")
    print("You do NOT need all_2010-2025_job_CLEANED_numeric_only_2.csv")
else:
    print("\n⚠️ Still needs checking.")
    if not bad_years.empty:
        print("\nBad years:")
        print(bad_years.to_string(index=False))

# ============================================================
# SAVE FINAL CHART/TABLE INPUT
# ============================================================

final_df.to_csv(OUTPUT_FILE, index=False)

print("\n" + "=" * 100)
print("SAVED")
print("=" * 100)
print(OUTPUT_FILE)

display(final_df.head(30))

# Fix code chart 6

In [ ]:
from pathlib import Path
import pandas as pd
import gc
import time

# ============================================================
# CREATE CLEANED _3 FROM CLEANED _1
#
# INPUT:
#   all_2010-2025_job_CLEANED_numeric_only_1.csv
#
# OUTPUT:
#   all_2010-2025_job_CLEANED_numeric_only_3.csv
#
# IMPORTANT:
# 1. Starts from CLEANED _1
# 2. Keeps ALL columns
# 3. Keeps ALL rows from 2010-2025
# 4. Does NOT filter major occupation groups only
# 5. Does NOT create job_major_occupation_FINAL file
# 6. Memory optimized with chunks
# ============================================================

INPUT_FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_1.csv"

OUTPUT_FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_3.csv"

CHUNKSIZE = 150_000

START_YEAR = 2010
END_YEAR = 2025

# Numeric columns to clean again safely if they exist
NUMERIC_COLS = [
    "year",
    "area",
    "area_type",
    "own_code",
    "tot_emp",
    "emp_prse",
    "jobs_1000",
    "loc_q",
    "loc_quotient",
    "pct_tot",
    "h_mean",
    "a_mean",
    "mean_prse",
    "h_pct10",
    "h_pct25",
    "h_median",
    "h_pct75",
    "h_pct90",
    "a_pct10",
    "a_pct25",
    "a_median",
    "a_pct75",
    "a_pct90",
    "annual",
    "hourly",
]

print("=" * 100)
print("CREATE CLEANED _3 FROM CLEANED _1")
print("=" * 100)
print(f"Input file:  {INPUT_FILE}")
print(f"Output file: {OUTPUT_FILE}")

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()
    print("\nOld _3 file found and removed.")

rows_seen = 0
rows_written = 0
chunk_count = 0
first_write = True
start_time = time.time()

print("\n" + "=" * 100)
print("READING CLEANED _1 + WRITING FULL CLEANED _3")
print("=" * 100)

for chunk_num, chunk in enumerate(
    pd.read_csv(INPUT_FILE, chunksize=CHUNKSIZE, low_memory=False),
    start=1
):
    chunk_count += 1
    rows_seen += len(chunk)

    # Standardize column names safely
    chunk.columns = (
        chunk.columns
        .astype(str)
        .str.strip()
        .str.lower()
    )

    if "year" not in chunk.columns:
        raise ValueError("Missing required column: year")

    # Clean year
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    # Keep 2010-2025 only
    chunk = chunk[
        (chunk["year"] >= START_YEAR) &
        (chunk["year"] <= END_YEAR)
    ].copy()

    # Clean numeric columns only if they exist
    for col in NUMERIC_COLS:
        if col in chunk.columns:
            chunk[col] = (
                chunk[col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .str.replace("$", "", regex=False)
                .str.replace("*", "", regex=False)
                .str.replace("#", "", regex=False)
                .str.strip()
            )

            chunk[col] = chunk[col].replace({
                "": pd.NA,
                "nan": pd.NA,
                "NaN": pd.NA,
                "None": pd.NA,
                "-": pd.NA,
                "**": pd.NA,
                "*": pd.NA,
                "#": pd.NA,
            })

            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    # Clean text/code columns but keep them as text
    TEXT_COLS = [
        "release",
        "source_workbook",
        "source_folder",
        "source_sheet",
        "source_period",
        "source_file",
        "source_path",
        "area_title",
        "prim_state",
        "state",
        "state_abbr",
        "naics",
        "naics_title",
        "i_group",
        "occ_code",
        "occ_title",
        "o_group",
    ]

    for col in TEXT_COLS:
        if col in chunk.columns:
            chunk[col] = chunk[col].astype("string").str.strip()

    # Write full chunk with ALL columns
    if len(chunk) > 0:
        chunk.to_csv(
            OUTPUT_FILE,
            mode="w" if first_write else "a",
            header=first_write,
            index=False
        )

        first_write = False
        rows_written += len(chunk)

    if chunk_num == 1 or chunk_num % 5 == 0:
        mins = (time.time() - start_time) / 60
        print(
            f"Chunk {chunk_num} done | rows seen: {rows_seen:,} | "
            f"rows written: {rows_written:,} | total min: {mins:.1f}",
            flush=True
        )

    del chunk
    gc.collect()

print("\n" + "=" * 100)
print("CLEANED _3 CREATED FROM CLEANED _1")
print("=" * 100)
print(f"Chunks processed: {chunk_count:,}")
print(f"Rows seen from _1: {rows_seen:,}")
print(f"Rows written _3:   {rows_written:,}")
print(f"Saved to:          {OUTPUT_FILE}")

# ============================================================
# CHECK SAVED _3 FILE
# ============================================================

print("\n" + "=" * 100)
print("CHECK SAVED _3 FILE")
print("=" * 100)

check_rows = 0
columns_seen = None
year_counts = {}

for chunk_num, chunk in enumerate(
    pd.read_csv(OUTPUT_FILE, chunksize=CHUNKSIZE, low_memory=False),
    start=1
):
    check_rows += len(chunk)

    if columns_seen is None:
        columns_seen = list(chunk.columns)

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    counts = chunk["year"].value_counts(dropna=False)

    for year_value, count_value in counts.items():
        year_counts[year_value] = year_counts.get(year_value, 0) + count_value

    del chunk
    gc.collect()

year_check = (
    pd.DataFrame({
        "year": list(year_counts.keys()),
        "row_count": list(year_counts.values())
    })
    .sort_values("year")
    .reset_index(drop=True)
)

print(f"Final rows:    {check_rows:,}")
print(f"Final columns: {len(columns_seen):,}")

print("\nColumns:")
for i, col in enumerate(columns_seen, start=1):
    print(f"{i:>3}. {col}")

print("\nRows by year:")
print(year_check.to_string(index=False))

print("\n" + "=" * 100)
print("DONE")
print("=" * 100)
print("✅ Full-column cleaned file created from _1:")
print(OUTPUT_FILE)

preview_df = pd.read_csv(OUTPUT_FILE, nrows=30, low_memory=False)
display(preview_df)

# new file for chart fix ver 3

In [ ]:
from pathlib import Path
import pandas as pd
import gc
import time
import re

# ============================================================
# CHECK ONLY: PROVE _3 IS SAFE FOR CHARTS
#
# NO SAVE
# NO DELETE
# NO CHANGE FILE
#
# This checks:
# 1. _1 and _3 files exist
# 2. _3 has same columns/order as _1
# 3. _3 has no duplicate column names
# 4. _3 row count matches _1 rows from 2010-2025
# 5. _3 year counts match _1 year counts
# 6. Important chart columns exist
# 7. Numeric columns are readable as numbers
# 8. Each chart has enough data
# 9. Prints clear final result you can copy/show
# ============================================================

SOURCE_FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_1.csv"
CHECK_FILE  = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_3.csv"

CHUNKSIZE = 150_000
START_YEAR = 2010
END_YEAR = 2025

IMPORTANT_COLS = [
    "year",
    "area",
    "area_title",
    "area_type",
    "prim_state",
    "state",
    "state_abbr",
    "naics",
    "naics_title",
    "i_group",
    "own_code",
    "occ_code",
    "occ_title",
    "o_group",
    "tot_emp",
    "h_median",
    "a_median",
]

NUMERIC_CHECK_COLS = [
    "year",
    "area",
    "area_type",
    "own_code",
    "tot_emp",
    "emp_prse",
    "jobs_1000",
    "loc_q",
    "loc_quotient",
    "pct_tot",
    "h_mean",
    "a_mean",
    "mean_prse",
    "h_pct10",
    "h_pct25",
    "h_median",
    "h_pct75",
    "h_pct90",
    "a_pct10",
    "a_pct25",
    "a_median",
    "a_pct75",
    "a_pct90",
    "annual",
    "hourly",
]

YEARS_EXPECTED = list(range(START_YEAR, END_YEAR + 1))


def clean_colnames(columns):
    return (
        pd.Index(columns)
        .astype(str)
        .str.strip()
        .str.lower()
        .tolist()
    )


def to_number(series):
    cleaned = (
        series
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.replace("#", "", regex=False)
        .str.strip()
    )

    cleaned = cleaned.replace({
        "": pd.NA,
        "nan": pd.NA,
        "NaN": pd.NA,
        "None": pd.NA,
        "-": pd.NA,
        "**": pd.NA,
        "*": pd.NA,
        "#": pd.NA,
    })

    return pd.to_numeric(cleaned, errors="coerce"), cleaned


def get_header_info(file_path, label):
    header_df = pd.read_csv(file_path, nrows=0, low_memory=False)
    columns = clean_colnames(header_df.columns)

    duplicate_columns = sorted({
        col for col in columns
        if columns.count(col) > 1
    })

    missing_important = [
        col for col in IMPORTANT_COLS
        if col not in columns
    ]

    return {
        "label": label,
        "columns": columns,
        "column_count": len(columns),
        "duplicate_columns": duplicate_columns,
        "missing_important": missing_important,
    }


def scan_year_only(file_path, label):
    print("\n" + "=" * 100)
    print(f"SCANNING YEAR COUNTS ONLY: {label}")
    print("=" * 100)

    rows_total = 0
    rows_in_year_range = 0
    year_counts = {}
    min_year = None
    max_year = None

    start_time = time.time()

    for chunk_num, chunk in enumerate(
        pd.read_csv(
            file_path,
            chunksize=CHUNKSIZE,
            low_memory=False,
            on_bad_lines="error",
            usecols=["year"]
        ),
        start=1
    ):
        chunk.columns = clean_colnames(chunk.columns)

        rows_total += len(chunk)

        year_num, _ = to_number(chunk["year"])

        if year_num.notna().any():
            chunk_min = int(year_num.min())
            chunk_max = int(year_num.max())
            min_year = chunk_min if min_year is None else min(min_year, chunk_min)
            max_year = chunk_max if max_year is None else max(max_year, chunk_max)

        in_range_mask = (
            (year_num >= START_YEAR) &
            (year_num <= END_YEAR)
        )

        rows_in_year_range += int(in_range_mask.sum())

        counts = year_num[in_range_mask].value_counts(dropna=False)

        for year_value, count_value in counts.items():
            year_counts[int(year_value)] = year_counts.get(int(year_value), 0) + int(count_value)

        if chunk_num == 1 or chunk_num % 5 == 0:
            mins = (time.time() - start_time) / 60
            print(
                f"{label} chunk {chunk_num} checked | rows seen: {rows_total:,} | min: {mins:.1f}",
                flush=True
            )

        del chunk
        gc.collect()

    year_check = (
        pd.DataFrame({
            "year": list(year_counts.keys()),
            "row_count": list(year_counts.values())
        })
        .sort_values("year")
        .reset_index(drop=True)
    )

    return {
        "label": label,
        "rows_total": rows_total,
        "rows_in_year_range": rows_in_year_range,
        "year_check": year_check,
        "min_year": min_year,
        "max_year": max_year,
    }


def scan_full_chart_readiness(file_path, label):
    print("\n" + "=" * 100)
    print(f"SCANNING FULL CHART READINESS: {label}")
    print("=" * 100)

    rows_total = 0

    numeric_bad_counts = {}
    numeric_notna_counts = {}

    year_has_rows = {}
    year_has_tot_emp = {}
    year_has_a_median = {}
    year_has_h_median = {}

    major_occ_rows = 0
    major_occ_with_emp = 0
    major_occ_with_wage = 0

    detailed_occ_rows = 0
    detailed_occ_with_emp = 0
    detailed_occ_with_wage = 0

    state_rows = 0
    industry_rows = 0
    area_rows = 0

    blank_counts = {
        "year_blank": 0,
        "occ_code_blank": 0,
        "occ_title_blank": 0,
        "tot_emp_blank_or_not_numeric": 0,
        "a_median_blank_or_not_numeric": 0,
        "h_median_blank_or_not_numeric": 0,
    }

    start_time = time.time()

    for chunk_num, chunk in enumerate(
        pd.read_csv(
            file_path,
            chunksize=CHUNKSIZE,
            low_memory=False,
            on_bad_lines="error"
        ),
        start=1
    ):
        rows_total += len(chunk)
        chunk.columns = clean_colnames(chunk.columns)

        year_num, year_clean = to_number(chunk["year"])
        tot_emp_num, tot_emp_clean = to_number(chunk["tot_emp"])
        a_median_num, a_median_clean = to_number(chunk["a_median"])
        h_median_num, h_median_clean = to_number(chunk["h_median"])

        year_in_range = (
            (year_num >= START_YEAR) &
            (year_num <= END_YEAR)
        )

        # Basic blank / nonnumeric checks
        blank_counts["year_blank"] += int(year_num.isna().sum())

        if "occ_code" in chunk.columns:
            occ_code = chunk["occ_code"].astype(str).str.strip()
            blank_counts["occ_code_blank"] += int(occ_code.isin(["", "nan", "NaN", "None"]).sum())
        else:
            occ_code = pd.Series([""] * len(chunk))

        if "occ_title" in chunk.columns:
            occ_title = chunk["occ_title"].astype(str).str.strip()
            blank_counts["occ_title_blank"] += int(occ_title.isin(["", "nan", "NaN", "None"]).sum())
        else:
            occ_title = pd.Series([""] * len(chunk))

        blank_counts["tot_emp_blank_or_not_numeric"] += int(tot_emp_num.isna().sum())
        blank_counts["a_median_blank_or_not_numeric"] += int(a_median_num.isna().sum())
        blank_counts["h_median_blank_or_not_numeric"] += int(h_median_num.isna().sum())

        # Numeric bad count check
        for col in NUMERIC_CHECK_COLS:
            if col in chunk.columns:
                numeric_version, cleaned = to_number(chunk[col])

                original_not_blank = cleaned.notna()
                bad_count = int((original_not_blank & numeric_version.isna()).sum())
                notna_count = int(numeric_version.notna().sum())

                numeric_bad_counts[col] = numeric_bad_counts.get(col, 0) + bad_count
                numeric_notna_counts[col] = numeric_notna_counts.get(col, 0) + notna_count

        # Year-level chart checks
        for year in YEARS_EXPECTED:
            ymask = year_num == year

            year_has_rows[year] = year_has_rows.get(year, 0) + int(ymask.sum())
            year_has_tot_emp[year] = year_has_tot_emp.get(year, 0) + int((ymask & tot_emp_num.notna()).sum())
            year_has_a_median[year] = year_has_a_median.get(year, 0) + int((ymask & a_median_num.notna()).sum())
            year_has_h_median[year] = year_has_h_median.get(year, 0) + int((ymask & h_median_num.notna()).sum())

        # Occupation checks
        occ_code_str = occ_code.astype(str).str.strip()

        major_mask = occ_code_str.str.match(r"^\d{2}-0000$", na=False)
        detailed_mask = (
            occ_code_str.str.match(r"^\d{2}-\d{4}$", na=False) &
            ~occ_code_str.str.match(r"^\d{2}-0000$", na=False)
        )

        major_occ_rows += int(major_mask.sum())
        major_occ_with_emp += int((major_mask & tot_emp_num.notna()).sum())
        major_occ_with_wage += int((major_mask & a_median_num.notna()).sum())

        detailed_occ_rows += int(detailed_mask.sum())
        detailed_occ_with_emp += int((detailed_mask & tot_emp_num.notna()).sum())
        detailed_occ_with_wage += int((detailed_mask & a_median_num.notna()).sum())

        # State / industry / area checks
        if "prim_state" in chunk.columns:
            prim_state = chunk["prim_state"].astype(str).str.strip()
            state_rows += int((prim_state.notna() & ~prim_state.isin(["", "nan", "NaN", "None"]) & tot_emp_num.notna()).sum())

        if "naics_title" in chunk.columns:
            naics_title = chunk["naics_title"].astype(str).str.strip()
            industry_rows += int((naics_title.notna() & ~naics_title.isin(["", "nan", "NaN", "None"]) & tot_emp_num.notna()).sum())

        if "area_title" in chunk.columns:
            area_title = chunk["area_title"].astype(str).str.strip()
            area_rows += int((area_title.notna() & ~area_title.isin(["", "nan", "NaN", "None"]) & tot_emp_num.notna()).sum())

        if chunk_num == 1 or chunk_num % 5 == 0:
            mins = (time.time() - start_time) / 60
            print(
                f"{label} chunk {chunk_num} checked | rows seen: {rows_total:,} | min: {mins:.1f}",
                flush=True
            )

        del chunk
        gc.collect()

    numeric_check = (
        pd.DataFrame({
            "column": list(numeric_bad_counts.keys()),
            "bad_numeric_count": [numeric_bad_counts[c] for c in numeric_bad_counts.keys()],
            "numeric_notna_count": [numeric_notna_counts[c] for c in numeric_bad_counts.keys()],
        })
        .sort_values("column")
        .reset_index(drop=True)
    )

    year_ready = pd.DataFrame({
        "year": YEARS_EXPECTED,
        "rows": [year_has_rows.get(y, 0) for y in YEARS_EXPECTED],
        "rows_with_tot_emp": [year_has_tot_emp.get(y, 0) for y in YEARS_EXPECTED],
        "rows_with_a_median": [year_has_a_median.get(y, 0) for y in YEARS_EXPECTED],
        "rows_with_h_median": [year_has_h_median.get(y, 0) for y in YEARS_EXPECTED],
    })

    chart_readiness = pd.DataFrame([
        {
            "chart": "Overall job count by year",
            "needed_data_rows": int(year_ready["rows_with_tot_emp"].sum()),
            "status": "OK" if (year_ready["rows_with_tot_emp"] > 0).all() else "PROBLEM"
        },
        {
            "chart": "Annual median wage by year",
            "needed_data_rows": int(year_ready["rows_with_a_median"].sum()),
            "status": "OK" if (year_ready["rows_with_a_median"] > 0).all() else "PROBLEM"
        },
        {
            "chart": "Top occupation groups by employment",
            "needed_data_rows": major_occ_with_emp,
            "status": "OK" if major_occ_with_emp > 0 else "PROBLEM"
        },
        {
            "chart": "Top detailed occupations by employment",
            "needed_data_rows": detailed_occ_with_emp,
            "status": "OK" if detailed_occ_with_emp > 0 else "PROBLEM"
        },
        {
            "chart": "Employment trend for top occupation groups",
            "needed_data_rows": major_occ_with_emp,
            "status": "OK" if major_occ_with_emp > 0 else "PROBLEM"
        },
        {
            "chart": "Annual median wage trend for top occupation groups",
            "needed_data_rows": major_occ_with_wage,
            "status": "OK" if major_occ_with_wage > 0 else "PROBLEM"
        },
        {
            "chart": "Top detailed occupations by annual median wage",
            "needed_data_rows": detailed_occ_with_wage,
            "status": "OK" if detailed_occ_with_wage > 0 else "PROBLEM"
        },
        {
            "chart": "Top states by employment",
            "needed_data_rows": state_rows,
            "status": "OK" if state_rows > 0 else "PROBLEM"
        },
        {
            "chart": "Top industries by employment",
            "needed_data_rows": industry_rows,
            "status": "OK" if industry_rows > 0 else "PROBLEM"
        },
        {
            "chart": "Top areas by employment",
            "needed_data_rows": area_rows,
            "status": "OK" if area_rows > 0 else "PROBLEM"
        },
        {
            "chart": "Hourly median wage by year",
            "needed_data_rows": int(year_ready["rows_with_h_median"].sum()),
            "status": "OK" if (year_ready["rows_with_h_median"] > 0).all() else "PROBLEM"
        },
    ])

    return {
        "rows_total": rows_total,
        "numeric_check": numeric_check,
        "year_ready": year_ready,
        "chart_readiness": chart_readiness,
        "blank_counts": blank_counts,
        "major_occ_rows": major_occ_rows,
        "major_occ_with_emp": major_occ_with_emp,
        "major_occ_with_wage": major_occ_with_wage,
        "detailed_occ_rows": detailed_occ_rows,
        "detailed_occ_with_emp": detailed_occ_with_emp,
        "detailed_occ_with_wage": detailed_occ_with_wage,
        "state_rows": state_rows,
        "industry_rows": industry_rows,
        "area_rows": area_rows,
    }


# ============================================================
# START CHECK
# ============================================================

print("=" * 100)
print("CHECK ONLY - SHOULD I USE VERSION 3 FOR CHARTS?")
print("=" * 100)

print(f"Source _1 file: {SOURCE_FILE}")
print(f"Check  _3 file: {CHECK_FILE}")

if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Source _1 file not found: {SOURCE_FILE}")

if not CHECK_FILE.exists():
    raise FileNotFoundError(f"Check _3 file not found: {CHECK_FILE}")

print("\nBoth files exist.")
print(f"_1 file size MB: {SOURCE_FILE.stat().st_size / 1024 / 1024:,.2f}")
print(f"_3 file size MB: {CHECK_FILE.stat().st_size / 1024 / 1024:,.2f}")

# ============================================================
# HEADER CHECK
# ============================================================

source_header = get_header_info(SOURCE_FILE, "SOURCE _1")
check_header = get_header_info(CHECK_FILE, "CHECK _3")

same_columns = source_header["columns"] == check_header["columns"]

print("\n" + "=" * 100)
print("COLUMN CHECK")
print("=" * 100)

print(f"_1 column count:    {source_header['column_count']}")
print(f"_3 column count:    {check_header['column_count']}")
print(f"Same columns/order: {same_columns}")

print("\nDuplicate columns in _3:")
print(check_header["duplicate_columns"])

print("\nMissing important chart columns in _3:")
print(check_header["missing_important"])

print("\nALL _3 COLUMNS:")
columns_df = pd.DataFrame({
    "column_number": range(1, len(check_header["columns"]) + 1),
    "column_name": check_header["columns"]
})
display(columns_df)

# ============================================================
# ROW / YEAR CHECK
# ============================================================

source_year_result = scan_year_only(SOURCE_FILE, "SOURCE _1")
check_year_result = scan_year_only(CHECK_FILE, "CHECK _3")

same_row_count = source_year_result["rows_in_year_range"] == check_year_result["rows_total"]

source_year = source_year_result["year_check"].copy()
check_year = check_year_result["year_check"].copy()

compare_year = source_year.merge(
    check_year,
    on="year",
    how="outer",
    suffixes=("_source_1", "_check_3")
).fillna(0)

compare_year["row_count_source_1"] = compare_year["row_count_source_1"].astype(int)
compare_year["row_count_check_3"] = compare_year["row_count_check_3"].astype(int)

compare_year["difference"] = (
    compare_year["row_count_check_3"] -
    compare_year["row_count_source_1"]
)

same_year_counts = (compare_year["difference"] == 0).all()

print("\n" + "=" * 100)
print("ROW COUNT CHECK")
print("=" * 100)

print(f"_1 rows in 2010-2025: {source_year_result['rows_in_year_range']:,}")
print(f"_3 total rows:         {check_year_result['rows_total']:,}")
print(f"Same row count:        {same_row_count}")

print(f"\n_3 year range:         {check_year_result['min_year']} to {check_year_result['max_year']}")
print(f"Same year counts:      {same_year_counts}")

print("\nRows by year comparison:")
display(compare_year)

# ============================================================
# FULL _3 CHART READINESS CHECK
# ============================================================

chart_result = scan_full_chart_readiness(CHECK_FILE, "CHECK _3")

print("\n" + "=" * 100)
print("YEAR DATA READINESS")
print("=" * 100)
display(chart_result["year_ready"])

print("\n" + "=" * 100)
print("CHART READINESS")
print("=" * 100)
display(chart_result["chart_readiness"])

print("\n" + "=" * 100)
print("NUMERIC CHECK")
print("=" * 100)

bad_numeric = chart_result["numeric_check"]
bad_numeric = bad_numeric[bad_numeric["bad_numeric_count"] > 0]

if len(bad_numeric) == 0:
    print("No bad numeric values found in checked numeric columns.")
else:
    print("Bad numeric values found:")
    display(bad_numeric)

print("\n" + "=" * 100)
print("BLANK / NONNUMERIC KEY VALUE COUNTS")
print("=" * 100)

blank_df = pd.DataFrame({
    "check": list(chart_result["blank_counts"].keys()),
    "count": list(chart_result["blank_counts"].values())
})
display(blank_df)

print("\n" + "=" * 100)
print("OCCUPATION / STATE / INDUSTRY / AREA ROW COUNTS")
print("=" * 100)

summary_df = pd.DataFrame([
    {"item": "major occupation rows using occ_code xx-0000", "count": chart_result["major_occ_rows"]},
    {"item": "major occupation rows with employment", "count": chart_result["major_occ_with_emp"]},
    {"item": "major occupation rows with annual median wage", "count": chart_result["major_occ_with_wage"]},
    {"item": "detailed occupation rows", "count": chart_result["detailed_occ_rows"]},
    {"item": "detailed occupation rows with employment", "count": chart_result["detailed_occ_with_emp"]},
    {"item": "detailed occupation rows with annual median wage", "count": chart_result["detailed_occ_with_wage"]},
    {"item": "state rows with employment", "count": chart_result["state_rows"]},
    {"item": "industry rows with employment", "count": chart_result["industry_rows"]},
    {"item": "area rows with employment", "count": chart_result["area_rows"]},
])
display(summary_df)

# ============================================================
# FINAL PROBLEM SUMMARY
# ============================================================

problems = []

if not same_columns:
    problems.append("_3 columns are not exactly the same as _1 columns/order.")

if check_header["duplicate_columns"]:
    problems.append("_3 has duplicate column names.")

if check_header["missing_important"]:
    problems.append("_3 is missing important chart columns.")

if not same_row_count:
    problems.append("_3 total row count does not match _1 rows from 2010-2025.")

if not same_year_counts:
    problems.append("_3 year counts do not match _1 year counts.")

if len(bad_numeric) > 0:
    problems.append("_3 has bad nonnumeric values in numeric columns.")

problem_charts = chart_result["chart_readiness"]
problem_charts = problem_charts[problem_charts["status"] != "OK"]

if len(problem_charts) > 0:
    problems.append("One or more planned charts may not have enough data.")

print("\n" + "=" * 100)
print("FINAL RESULT")
print("=" * 100)

if len(problems) == 0:
    print("CLEAN CHECK PASSED.")
    print("Use version 3 for chart making:")
    print(CHECK_FILE)
else:
    print("CHECK FOUND POSSIBLE PROBLEMS:")
    for p in problems:
        print(f"- {p}")

print("\nWhat to copy/show me if you want me to verify:")
print("1. FINAL RESULT")
print("2. ROW COUNT CHECK")
print("3. Rows by year comparison")
print("4. COLUMN CHECK")
print("5. CHART READINESS")
print("6. NUMERIC CHECK")
print("7. BLANK / NONNUMERIC KEY VALUE COUNTS")

# ============================================================
# PREVIEW ONLY
# ============================================================

print("\n" + "=" * 100)
print("PREVIEW _3 FIRST 30 ROWS")
print("=" * 100)

preview_df = pd.read_csv(CHECK_FILE, nrows=30, low_memory=False)
display(preview_df)

# ver 3 chart - read column

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# READ ONLY - PREVIEW FIRST 5 ROWS
# NO SAVE
# NO CHART
# NO FILE CHANGE
# ============================================================

FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_3.csv"

print("=" * 100)
print("READING FIRST 5 ROWS ONLY")
print("=" * 100)
print("File:", FILE)

preview = pd.read_csv(FILE, nrows=5)

print()
print("Shape preview:", preview.shape)
print()
display(preview)

print()
print("=" * 100)
print("DONE - preview only, no file saved or changed")
print("=" * 100)

# ver 3 chart - make chart

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import gc
import time

# ============================================================
# FIX CHART 05 AND 06
# USE FILE _3
# OUTPUT FOLDER _3
#
# IMPORTANT:
# 2025 is used ONLY to choose the top occupation groups.
# Final chart/table uses ALL YEARS 2010-2025.
# ============================================================

INPUT_FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_3.csv"

DOWNLOADS = Path.home() / "Downloads"
CHART_DIR = DOWNLOADS / "Job_Chart_3"
TABLE_DIR = DOWNLOADS / "Job_Table_3"

CHART_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

TOP_N = 10
CHUNK_SIZE = 150_000

print("=" * 100)
print("FIXING CHART 05 AND 06 - 2010 TO 2025")
print("=" * 100)
print("Input file:", INPUT_FILE)
print("Chart folder:", CHART_DIR)
print("Table folder:", TABLE_DIR)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File not found: {INPUT_FILE}")

# ------------------------------------------------------------
# Read columns only first
# ------------------------------------------------------------

all_columns = pd.read_csv(INPUT_FILE, nrows=0).columns.tolist()

needed_cols = [
    "year",
    "occ_code",
    "occ_title",
    "tot_emp",
    "a_median",
]

usecols = [c for c in needed_cols if c in all_columns]

missing_cols = [c for c in needed_cols if c not in all_columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print()
print("Using columns:", usecols)

# ------------------------------------------------------------
# Read only needed rows/columns with memory optimization
# Keep major occupation groups only: xx-0000
# Exclude 00-0000 because that is all occupations, not a group
# ------------------------------------------------------------

pieces = []
start = time.time()
rows_seen = 0
rows_kept = 0

for chunk_num, chunk in enumerate(
    pd.read_csv(INPUT_FILE, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False),
    start=1
):
    rows_seen += len(chunk)

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["occ_code"] = chunk["occ_code"].astype(str).str.strip()
    chunk["occ_title"] = chunk["occ_title"].astype(str).str.strip()

    chunk["tot_emp"] = (
        chunk["tot_emp"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.strip()
    )
    chunk["tot_emp"] = pd.to_numeric(chunk["tot_emp"], errors="coerce")

    chunk["a_median"] = (
        chunk["a_median"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.strip()
    )
    chunk["a_median"] = pd.to_numeric(chunk["a_median"], errors="coerce")

    mask = (
        chunk["year"].between(2010, 2025)
        & chunk["occ_code"].str.match(r"^\d{2}-0000$", na=False)
        & (chunk["occ_code"] != "00-0000")
    )

    kept = chunk.loc[mask, ["year", "occ_code", "occ_title", "tot_emp", "a_median"]].copy()

    if not kept.empty:
        pieces.append(kept)
        rows_kept += len(kept)

    if chunk_num == 1 or chunk_num % 5 == 0:
        elapsed = (time.time() - start) / 60
        print(
            f"Chunk {chunk_num} done | rows seen: {rows_seen:,} | "
            f"major group rows kept: {rows_kept:,} | minutes: {elapsed:.1f}",
            flush=True
        )

    del chunk, kept
    gc.collect()

if not pieces:
    raise ValueError("No major occupation group rows found.")

df = pd.concat(pieces, ignore_index=True)
del pieces
gc.collect()

df["year"] = df["year"].astype(int)

print()
print("=" * 100)
print("MAJOR OCCUPATION GROUP DATA LOADED")
print("=" * 100)
print("Rows:", len(df))
print("Year min:", df["year"].min())
print("Year max:", df["year"].max())
print("Unique years:", sorted(df["year"].dropna().unique().tolist()))
print("Unique occupation groups:", df["occ_code"].nunique())

# ------------------------------------------------------------
# Deduplicate correctly
# Do NOT sum duplicates.
# Keep one best row per year + occ_code by highest employment.
# This avoids 2010/2012 o_group/i_group missing problems.
# ------------------------------------------------------------

df_sorted = df.sort_values(
    ["year", "occ_code", "tot_emp"],
    ascending=[True, True, False],
    na_position="last"
)

best_rows = (
    df_sorted
    .drop_duplicates(subset=["year", "occ_code"], keep="first")
    .copy()
)

del df_sorted
gc.collect()

print()
print("=" * 100)
print("AFTER DEDUPLICATION")
print("=" * 100)
print("Rows:", len(best_rows))
print("Year min:", best_rows["year"].min())
print("Year max:", best_rows["year"].max())
print("Unique years:", sorted(best_rows["year"].unique().tolist()))

# ------------------------------------------------------------
# Pick top occupation groups from 2025 only
# Then chart those same groups for 2010-2025
# ------------------------------------------------------------

latest_year = 2025

top_groups_2025 = (
    best_rows[
        (best_rows["year"] == latest_year)
        & best_rows["tot_emp"].notna()
    ]
    .sort_values("tot_emp", ascending=False)
    .head(TOP_N)
    [["occ_code", "occ_title", "tot_emp"]]
    .copy()
)

if top_groups_2025.empty:
    raise ValueError("No 2025 major occupation group rows found. Cannot choose top groups.")

top_codes = top_groups_2025["occ_code"].tolist()

print()
print("=" * 100)
print(f"TOP {TOP_N} OCCUPATION GROUPS SELECTED FROM 2025")
print("=" * 100)
print(top_groups_2025.to_string(index=False))

# Better title map
title_map = (
    best_rows[best_rows["occ_code"].isin(top_codes)]
    .dropna(subset=["occ_title"])
    .sort_values(["occ_code", "year"])
    .drop_duplicates(subset=["occ_code"], keep="last")
    .set_index("occ_code")["occ_title"]
    .to_dict()
)

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def safe_file_name(name):
    bad = ['/', '\\', ':', '*', '?', '"', '<', '>', '|']
    for b in bad:
        name = name.replace(b, "")
    return name.replace("  ", " ").strip()


def save_table(df, file_stem):
    path = TABLE_DIR / f"{safe_file_name(file_stem)}.csv"
    df.to_csv(path, index=False)
    print("Saved table:", path)
    return path


def save_multi_line_chart(
    df,
    x_col,
    y_col,
    group_col,
    title,
    xlabel,
    ylabel,
    file_stem,
    money=False
):
    plt.figure(figsize=(14, 8))

    for group_name, group_df in df.groupby(group_col):
        group_df = group_df.sort_values(x_col)
        plt.plot(
            group_df[x_col],
            group_df[y_col],
            marker="o",
            linewidth=2,
            label=group_name
        )

    plt.title(title, fontsize=16, fontweight="bold")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(2010, 2026), rotation=45)
    plt.grid(True, alpha=0.3)

    if money:
        current_values = plt.gca().get_yticks()
        plt.gca().set_yticklabels([f"${x:,.0f}" for x in current_values])
    else:
        current_values = plt.gca().get_yticks()
        plt.gca().set_yticklabels([f"{x:,.0f}" for x in current_values])

    plt.legend(
        title="Occupation group",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9
    )

    plt.tight_layout()

    path = CHART_DIR / f"{safe_file_name(file_stem)}.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print("Saved chart:", path)
    return path

# ============================================================
# CHART 05
# Employment trend for top occupation groups
# ============================================================

file_stem_05 = "05_Employment_trend_for_top_occupation_groups"

table_05_fixed = (
    best_rows[
        best_rows["occ_code"].isin(top_codes)
        & best_rows["tot_emp"].notna()
    ]
    .copy()
)

table_05_fixed["occupation_group"] = table_05_fixed["occ_code"].map(title_map)

table_05_fixed = (
    table_05_fixed
    .rename(columns={"tot_emp": "employment"})
    [["year", "occ_code", "occupation_group", "employment"]]
    .sort_values(["occupation_group", "year"])
)

print()
print("=" * 100)
print("CHART 05 YEAR CHECK")
print("=" * 100)
print("Min year:", table_05_fixed["year"].min())
print("Max year:", table_05_fixed["year"].max())
print("Years included:", sorted(table_05_fixed["year"].unique().tolist()))
print()
print(table_05_fixed.groupby("year").size().to_string())

save_table(table_05_fixed, file_stem_05)

save_multi_line_chart(
    table_05_fixed,
    x_col="year",
    y_col="employment",
    group_col="occupation_group",
    title="Employment Trend for Top Occupation Groups, 2010-2025",
    xlabel="Year",
    ylabel="Employment",
    file_stem=file_stem_05,
    money=False
)

# ============================================================
# CHART 06
# Annual median wage trend for top occupation groups
# ============================================================

file_stem_06 = "06_Annual_median_wage_trend_for_top_occupation_groups"

table_06_fixed = (
    best_rows[
        best_rows["occ_code"].isin(top_codes)
        & best_rows["a_median"].notna()
    ]
    .copy()
)

table_06_fixed["occupation_group"] = table_06_fixed["occ_code"].map(title_map)

table_06_fixed = (
    table_06_fixed
    .rename(columns={"a_median": "annual_median_wage"})
    [["year", "occ_code", "occupation_group", "annual_median_wage"]]
    .sort_values(["occupation_group", "year"])
)

print()
print("=" * 100)
print("CHART 06 YEAR CHECK")
print("=" * 100)
print("Min year:", table_06_fixed["year"].min())
print("Max year:", table_06_fixed["year"].max())
print("Years included:", sorted(table_06_fixed["year"].unique().tolist()))
print()
print(table_06_fixed.groupby("year").size().to_string())

save_table(table_06_fixed, file_stem_06)

save_multi_line_chart(
    table_06_fixed,
    x_col="year",
    y_col="annual_median_wage",
    group_col="occupation_group",
    title="Annual Median Wage Trend for Top Occupation Groups, 2010-2025",
    xlabel="Year",
    ylabel="Annual median wage",
    file_stem=file_stem_06,
    money=True
)

print()
print("=" * 100)
print("DONE - CHART 05 AND 06 FIXED")
print("=" * 100)
print("Saved to:")
print("Charts:", CHART_DIR)
print("Tables:", TABLE_DIR)

# ver 3 chart - all chart

This should give you:
1. Overall job count by year
2. Annual median wage by year
3. Top occupation groups by employment
4. Top detailed occupations by employment
5. Employment trend for top occupation groups
6. Annual median wage trend for top occupation groups
7. Top detailed occupations by annual median wage
8. Top states by employment
9. Top industries by employment
10. Top areas by employment
11. Hourly median wage by year

# Important: charts 05, 06, 01, 02, and 11 are true 2010-2025 trend charts. Charts 03, 04, 07, 08, 09, and 10 are latest-year ranking charts using 2025.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import gc
import time

# ============================================================
# MAKE ALL JOB CHARTS FROM CLEANED _3 FILE
#
# Input:
#   Downloads/all_2010-2025_job_CLEANED_numeric_only_3.csv
#
# Output:
#   Downloads/Job_Chart_3
#   Downloads/Job_Table_3
#
# IMPORTANT:
# - This does NOT create a new cleaned CSV.
# - It only reads _3 and creates charts/tables.
# - Chart 05 and 06 use all years 2010-2025.
# - Top snapshot charts use 2025 because 2025 is the latest year.
# ============================================================

INPUT_FILE = Path.home() / "Downloads" / "all_2010-2025_job_CLEANED_numeric_only_3.csv"

DOWNLOADS = Path.home() / "Downloads"
CHART_DIR = DOWNLOADS / "Job_Chart_3"
TABLE_DIR = DOWNLOADS / "Job_Table_3"

CHART_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

TOP_N = 10
CHUNK_SIZE = 150_000
LATEST_YEAR = 2025
SHOW_PLOTS = True

print("=" * 100)
print("MAKE ALL JOB CHARTS - 2010 TO 2025")
print("=" * 100)
print("Input file:", INPUT_FILE)
print("Chart folder:", CHART_DIR)
print("Table folder:", TABLE_DIR)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File not found: {INPUT_FILE}")

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def safe_file_name(name):
    bad = ['/', '\\', ':', '*', '?', '"', '<', '>', '|']
    for b in bad:
        name = name.replace(b, "")
    return name.replace("  ", " ").strip()


def clean_number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.replace("#", "", regex=False)
        .str.strip(),
        errors="coerce"
    )


def save_table(df, file_stem):
    path = TABLE_DIR / f"{safe_file_name(file_stem)}.csv"
    df.to_csv(path, index=False)
    print("Saved table:", path)
    return path


def money_formatter(x, pos):
    return f"${x:,.0f}"


def number_formatter(x, pos):
    return f"{x:,.0f}"


def shorten_label(x, max_len=60):
    x = str(x)
    if len(x) <= max_len:
        return x
    return x[:max_len - 3] + "..."


def save_line_chart(df, x_col, y_col, title, xlabel, ylabel, file_stem, money=False):
    if df.empty:
        print("SKIP chart, empty data:", file_stem)
        return None

    plt.figure(figsize=(14, 7))
    plt.plot(df[x_col], df[y_col], marker="o", linewidth=2)

    plt.title(title, fontsize=16, fontweight="bold")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(2010, 2026), rotation=45)
    plt.grid(True, alpha=0.3)

    ax = plt.gca()
    ax.yaxis.set_major_formatter(FuncFormatter(money_formatter if money else number_formatter))

    plt.tight_layout()

    path = CHART_DIR / f"{safe_file_name(file_stem)}.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")

    if SHOW_PLOTS:
        plt.show()

    plt.close()
    print("Saved chart:", path)
    return path


def save_barh_chart(df, label_col, value_col, title, xlabel, file_stem, money=False):
    if df.empty:
        print("SKIP chart, empty data:", file_stem)
        return None

    plot_df = df.copy()
    plot_df["plot_label"] = plot_df[label_col].apply(shorten_label)

    # For horizontal bar chart, smallest first so biggest appears on top
    plot_df = plot_df.sort_values(value_col, ascending=True)

    plt.figure(figsize=(14, 8))
    plt.barh(plot_df["plot_label"], plot_df[value_col])

    plt.title(title, fontsize=16, fontweight="bold")
    plt.xlabel(xlabel)
    plt.ylabel("")
    plt.grid(True, axis="x", alpha=0.3)

    ax = plt.gca()
    ax.xaxis.set_major_formatter(FuncFormatter(money_formatter if money else number_formatter))

    plt.tight_layout()

    path = CHART_DIR / f"{safe_file_name(file_stem)}.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")

    if SHOW_PLOTS:
        plt.show()

    plt.close()
    print("Saved chart:", path)
    return path


def save_multi_line_chart(df, x_col, y_col, group_col, title, xlabel, ylabel, file_stem, money=False):
    if df.empty:
        print("SKIP chart, empty data:", file_stem)
        return None

    plt.figure(figsize=(14, 8))

    for group_name, group_df in df.groupby(group_col):
        group_df = group_df.sort_values(x_col)
        plt.plot(
            group_df[x_col],
            group_df[y_col],
            marker="o",
            linewidth=2,
            label=shorten_label(group_name, 45)
        )

    plt.title(title, fontsize=16, fontweight="bold")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(2010, 2026), rotation=45)
    plt.grid(True, alpha=0.3)

    ax = plt.gca()
    ax.yaxis.set_major_formatter(FuncFormatter(money_formatter if money else number_formatter))

    plt.legend(
        title="Occupation group",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9
    )

    plt.tight_layout()

    path = CHART_DIR / f"{safe_file_name(file_stem)}.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")

    if SHOW_PLOTS:
        plt.show()

    plt.close()
    print("Saved chart:", path)
    return path


def concat_or_empty(pieces):
    if pieces:
        return pd.concat(pieces, ignore_index=True)
    return pd.DataFrame()


def dedupe_keep_highest_employment(df, keys):
    if df.empty:
        return df.copy()

    return (
        df.sort_values(
            keys + ["tot_emp"],
            ascending=[True] * len(keys) + [False],
            na_position="last"
        )
        .drop_duplicates(subset=keys, keep="first")
        .copy()
    )


def text_mask(df, col, pattern):
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    return df[col].astype(str).str.contains(pattern, case=False, na=False, regex=True)


# ------------------------------------------------------------
# Read only needed columns
# ------------------------------------------------------------

all_columns = pd.read_csv(INPUT_FILE, nrows=0).columns.tolist()

wanted_cols = [
    "year",
    "occ_code",
    "occ_title",
    "tot_emp",
    "a_median",
    "h_median",
    "area",
    "area_title",
    "area_type",
    "prim_state",
    "state",
    "state_abbr",
    "naics",
    "naics_title",
    "i_group",
    "own_code",
    "data_level",
]

usecols = [c for c in wanted_cols if c in all_columns]

required_cols = ["year", "occ_code", "occ_title", "tot_emp"]
missing_required = [c for c in required_cols if c not in usecols]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print()
print("Columns used:")
print(usecols)

# ------------------------------------------------------------
# One memory-optimized pass through the CSV
# ------------------------------------------------------------

overall_pieces = []
major_group_pieces = []
detail_2025_pieces = []
state_2025_pieces = []
industry_2025_pieces = []
area_2025_pieces = []

start = time.time()
rows_seen = 0

for chunk_num, chunk in enumerate(
    pd.read_csv(INPUT_FILE, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False),
    start=1
):
    rows_seen += len(chunk)

    # Clean basic fields
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    for col in [
        "occ_code",
        "occ_title",
        "area",
        "area_title",
        "area_type",
        "prim_state",
        "state",
        "state_abbr",
        "naics",
        "naics_title",
        "i_group",
        "own_code",
        "data_level",
    ]:
        if col in chunk.columns:
            chunk[col] = chunk[col].astype(str).str.strip()

    for col in ["tot_emp", "a_median", "h_median"]:
        if col in chunk.columns:
            chunk[col] = clean_number(chunk[col])

    year_ok = chunk["year"].between(2010, 2025)
    year_2025 = chunk["year"].eq(LATEST_YEAR)

    occ_code = chunk["occ_code"].astype(str)

    all_occupations = occ_code.eq("00-0000")

    major_group = (
        occ_code.str.match(r"^\d{2}-0000$", na=False)
        & ~all_occupations
    )

    detailed_occupation = (
        occ_code.str.match(r"^\d{2}-\d{4}$", na=False)
        & ~occ_code.str.endswith("-0000", na=False)
        & ~all_occupations
    )

    # 01, 02, 11: overall rows, then later keep biggest employment per year
    overall_mask = year_ok & all_occupations
    if overall_mask.any():
        keep_cols = [c for c in [
            "year", "occ_code", "occ_title", "tot_emp", "a_median", "h_median",
            "area_title", "data_level", "naics_title"
        ] if c in chunk.columns]
        overall_pieces.append(chunk.loc[overall_mask, keep_cols].copy())

    # 03, 05, 06: major occupation groups
    major_mask = year_ok & major_group
    if major_mask.any():
        keep_cols = [c for c in [
            "year", "occ_code", "occ_title", "tot_emp", "a_median"
        ] if c in chunk.columns]
        major_group_pieces.append(chunk.loc[major_mask, keep_cols].copy())

    # 04, 07: detailed occupations, latest year only
    detail_mask = year_2025 & detailed_occupation
    if detail_mask.any():
        keep_cols = [c for c in [
            "year", "occ_code", "occ_title", "tot_emp", "a_median"
        ] if c in chunk.columns]
        detail_2025_pieces.append(chunk.loc[detail_mask, keep_cols].copy())

    # 08: states, latest year only
    if "data_level" in chunk.columns:
        state_level_mask = chunk["data_level"].str.lower().eq("state")
    elif "area_type" in chunk.columns:
        state_level_mask = pd.to_numeric(chunk["area_type"], errors="coerce").eq(2)
    else:
        state_level_mask = (
            chunk.get("state_abbr", pd.Series("", index=chunk.index)).astype(str).ne("")
            & ~text_mask(chunk, "area_title", "metropolitan|nonmetropolitan")
        )

    state_mask = year_2025 & all_occupations & state_level_mask
    if state_mask.any():
        keep_cols = [c for c in [
            "year", "area_title", "state", "state_abbr", "tot_emp", "a_median", "h_median", "data_level"
        ] if c in chunk.columns]
        state_2025_pieces.append(chunk.loc[state_mask, keep_cols].copy())

    # 09: industries, latest year only
    if "data_level" in chunk.columns:
        industry_level_mask = chunk["data_level"].str.lower().str.contains("industry", na=False)
    elif "i_group" in chunk.columns:
        industry_level_mask = chunk["i_group"].astype(str).str.strip().ne("")
    else:
        industry_level_mask = pd.Series(False, index=chunk.index)

    if "naics_title" in chunk.columns:
        industry_level_mask = (
            industry_level_mask
            & ~chunk["naics_title"].str.lower().str.contains("all industries|cross-industry", na=False)
        )

    industry_mask = year_2025 & all_occupations & industry_level_mask
    if industry_mask.any():
        keep_cols = [c for c in [
            "year", "naics", "naics_title", "i_group", "tot_emp", "a_median", "h_median", "data_level"
        ] if c in chunk.columns]
        industry_2025_pieces.append(chunk.loc[industry_mask, keep_cols].copy())

    # 10: metro / nonmetro areas, latest year only
    if "data_level" in chunk.columns:
        area_level_mask = chunk["data_level"].str.lower().str.contains("metro|nonmetro", na=False)
    elif "area_title" in chunk.columns:
        area_level_mask = chunk["area_title"].str.lower().str.contains("metropolitan|nonmetropolitan", na=False)
    else:
        area_level_mask = pd.Series(False, index=chunk.index)

    area_mask = year_2025 & all_occupations & area_level_mask
    if area_mask.any():
        keep_cols = [c for c in [
            "year", "area", "area_title", "state", "state_abbr", "tot_emp", "a_median", "h_median", "data_level"
        ] if c in chunk.columns]
        area_2025_pieces.append(chunk.loc[area_mask, keep_cols].copy())

    if chunk_num == 1 or chunk_num % 5 == 0:
        elapsed = (time.time() - start) / 60
        print(
            f"Chunk {chunk_num} done | rows seen: {rows_seen:,} | minutes: {elapsed:.1f}",
            flush=True
        )

    del chunk
    gc.collect()

print()
print("=" * 100)
print("READING DONE")
print("=" * 100)

# ------------------------------------------------------------
# Combine small kept pieces
# ------------------------------------------------------------

overall_df = concat_or_empty(overall_pieces)
major_df = concat_or_empty(major_group_pieces)
detail_2025_df = concat_or_empty(detail_2025_pieces)
state_2025_df = concat_or_empty(state_2025_pieces)
industry_2025_df = concat_or_empty(industry_2025_pieces)
area_2025_df = concat_or_empty(area_2025_pieces)

del overall_pieces, major_group_pieces, detail_2025_pieces
del state_2025_pieces, industry_2025_pieces, area_2025_pieces
gc.collect()

print("Overall candidate rows:", len(overall_df))
print("Major group candidate rows:", len(major_df))
print("Detailed occupation 2025 candidate rows:", len(detail_2025_df))
print("State 2025 candidate rows:", len(state_2025_df))
print("Industry 2025 candidate rows:", len(industry_2025_df))
print("Area 2025 candidate rows:", len(area_2025_df))

# ------------------------------------------------------------
# Deduplicate correctly
# DO NOT sum duplicates.
# Keep the row with highest employment for each key.
# ------------------------------------------------------------

overall_best = dedupe_keep_highest_employment(overall_df, ["year"])
major_best = dedupe_keep_highest_employment(major_df, ["year", "occ_code"])
detail_2025_best = dedupe_keep_highest_employment(detail_2025_df, ["occ_code"])

print()
print("=" * 100)
print("DEDUP CHECK")
print("=" * 100)

if not overall_best.empty:
    print("Overall years:", sorted(overall_best["year"].dropna().astype(int).unique().tolist()))

if not major_best.empty:
    print("Major group years:", sorted(major_best["year"].dropna().astype(int).unique().tolist()))
    print("Major groups:", major_best["occ_code"].nunique())

# ------------------------------------------------------------
# CHART 01
# Overall job count by year
# ------------------------------------------------------------

file_stem = "01_Overall_job_count_by_year"

table_01 = (
    overall_best[["year", "tot_emp"]]
    .dropna(subset=["year", "tot_emp"])
    .rename(columns={"tot_emp": "employment"})
    .sort_values("year")
)

table_01["year"] = table_01["year"].astype(int)

save_table(table_01, file_stem)

save_line_chart(
    table_01,
    x_col="year",
    y_col="employment",
    title="Overall Job Count by Year, 2010-2025",
    xlabel="Year",
    ylabel="Employment",
    file_stem=file_stem,
    money=False
)

# ------------------------------------------------------------
# CHART 02
# Annual median wage by year
# ------------------------------------------------------------

file_stem = "02_Annual_median_wage_by_year"

if "a_median" in overall_best.columns:
    table_02 = (
        overall_best[["year", "a_median"]]
        .dropna(subset=["year", "a_median"])
        .rename(columns={"a_median": "annual_median_wage"})
        .sort_values("year")
    )

    table_02["year"] = table_02["year"].astype(int)

    save_table(table_02, file_stem)

    save_line_chart(
        table_02,
        x_col="year",
        y_col="annual_median_wage",
        title="Annual Median Wage by Year, 2010-2025",
        xlabel="Year",
        ylabel="Annual median wage",
        file_stem=file_stem,
        money=True
    )
else:
    print("SKIP 02 because a_median column is missing.")

# ------------------------------------------------------------
# CHART 03
# Top occupation groups by employment, 2025
# ------------------------------------------------------------

file_stem = "03_Top_occupation_groups_by_employment"

table_03 = (
    major_best[
        (major_best["year"] == LATEST_YEAR)
        & major_best["tot_emp"].notna()
    ]
    .sort_values("tot_emp", ascending=False)
    .head(TOP_N)
    .rename(columns={
        "occ_title": "occupation_group",
        "tot_emp": "employment"
    })
    [["year", "occ_code", "occupation_group", "employment"]]
)

save_table(table_03, file_stem)

save_barh_chart(
    table_03,
    label_col="occupation_group",
    value_col="employment",
    title=f"Top {TOP_N} Occupation Groups by Employment, {LATEST_YEAR}",
    xlabel="Employment",
    file_stem=file_stem,
    money=False
)

# ------------------------------------------------------------
# CHART 04
# Top detailed occupations by employment, 2025
# ------------------------------------------------------------

file_stem = "04_Top_detailed_occupations_by_employment"

table_04 = (
    detail_2025_best[detail_2025_best["tot_emp"].notna()]
    .sort_values("tot_emp", ascending=False)
    .head(TOP_N)
    .rename(columns={
        "occ_title": "detailed_occupation",
        "tot_emp": "employment"
    })
    [["year", "occ_code", "detailed_occupation", "employment"]]
)

save_table(table_04, file_stem)

save_barh_chart(
    table_04,
    label_col="detailed_occupation",
    value_col="employment",
    title=f"Top {TOP_N} Detailed Occupations by Employment, {LATEST_YEAR}",
    xlabel="Employment",
    file_stem=file_stem,
    money=False
)

# ------------------------------------------------------------
# CHART 05
# Employment trend for top occupation groups, 2010-2025
# Pick top groups from 2025, then show all years.
# ------------------------------------------------------------

file_stem = "05_Employment_trend_for_top_occupation_groups"

top_groups_2025 = (
    major_best[
        (major_best["year"] == LATEST_YEAR)
        & major_best["tot_emp"].notna()
    ]
    .sort_values("tot_emp", ascending=False)
    .head(TOP_N)
    [["occ_code", "occ_title", "tot_emp"]]
    .copy()
)

top_codes = top_groups_2025["occ_code"].tolist()

title_map = (
    major_best[major_best["occ_code"].isin(top_codes)]
    .dropna(subset=["occ_title"])
    .sort_values(["occ_code", "year"])
    .drop_duplicates(subset=["occ_code"], keep="last")
    .set_index("occ_code")["occ_title"]
    .to_dict()
)

table_05 = (
    major_best[
        major_best["occ_code"].isin(top_codes)
        & major_best["tot_emp"].notna()
    ]
    .copy()
)

table_05["occupation_group"] = table_05["occ_code"].map(title_map)

table_05 = (
    table_05
    .rename(columns={"tot_emp": "employment"})
    [["year", "occ_code", "occupation_group", "employment"]]
    .sort_values(["occupation_group", "year"])
)

table_05["year"] = table_05["year"].astype(int)

print()
print("=" * 100)
print("CHART 05 YEAR CHECK")
print("=" * 100)
print("Years included:", sorted(table_05["year"].unique().tolist()))

save_table(table_05, file_stem)

save_multi_line_chart(
    table_05,
    x_col="year",
    y_col="employment",
    group_col="occupation_group",
    title="Employment Trend for Top Occupation Groups, 2010-2025",
    xlabel="Year",
    ylabel="Employment",
    file_stem=file_stem,
    money=False
)

# ------------------------------------------------------------
# CHART 06
# Annual median wage trend for top occupation groups, 2010-2025
# ------------------------------------------------------------

file_stem = "06_Annual_median_wage_trend_for_top_occupation_groups"

if "a_median" in major_best.columns:
    table_06 = (
        major_best[
            major_best["occ_code"].isin(top_codes)
            & major_best["a_median"].notna()
        ]
        .copy()
    )

    table_06["occupation_group"] = table_06["occ_code"].map(title_map)

    table_06 = (
        table_06
        .rename(columns={"a_median": "annual_median_wage"})
        [["year", "occ_code", "occupation_group", "annual_median_wage"]]
        .sort_values(["occupation_group", "year"])
    )

    table_06["year"] = table_06["year"].astype(int)

    print()
    print("=" * 100)
    print("CHART 06 YEAR CHECK")
    print("=" * 100)
    print("Years included:", sorted(table_06["year"].unique().tolist()))

    save_table(table_06, file_stem)

    save_multi_line_chart(
        table_06,
        x_col="year",
        y_col="annual_median_wage",
        group_col="occupation_group",
        title="Annual Median Wage Trend for Top Occupation Groups, 2010-2025",
        xlabel="Year",
        ylabel="Annual median wage",
        file_stem=file_stem,
        money=True
    )
else:
    print("SKIP 06 because a_median column is missing.")

# ------------------------------------------------------------
# CHART 07
# Top detailed occupations by annual median wage, 2025
# ------------------------------------------------------------

file_stem = "07_Top_detailed_occupations_by_annual_median_wage"

if "a_median" in detail_2025_best.columns:
    table_07 = (
        detail_2025_best[
            detail_2025_best["a_median"].notna()
            & detail_2025_best["tot_emp"].notna()
        ]
        .sort_values("a_median", ascending=False)
        .head(TOP_N)
        .rename(columns={
            "occ_title": "detailed_occupation",
            "a_median": "annual_median_wage",
            "tot_emp": "employment"
        })
        [["year", "occ_code", "detailed_occupation", "employment", "annual_median_wage"]]
    )

    save_table(table_07, file_stem)

    save_barh_chart(
        table_07,
        label_col="detailed_occupation",
        value_col="annual_median_wage",
        title=f"Top {TOP_N} Detailed Occupations by Annual Median Wage, {LATEST_YEAR}",
        xlabel="Annual median wage",
        file_stem=file_stem,
        money=True
    )
else:
    print("SKIP 07 because a_median column is missing.")

# ------------------------------------------------------------
# CHART 08
# Top states by employment, 2025
# ------------------------------------------------------------

file_stem = "08_Top_states_by_employment"

if not state_2025_df.empty:
    if "area_title" in state_2025_df.columns:
        state_2025_df["state_name"] = state_2025_df["area_title"]
    elif "state_abbr" in state_2025_df.columns:
        state_2025_df["state_name"] = state_2025_df["state_abbr"]
    else:
        state_2025_df["state_name"] = "Unknown"

    state_best = dedupe_keep_highest_employment(state_2025_df, ["state_name"])

    table_08 = (
        state_best[state_best["tot_emp"].notna()]
        .sort_values("tot_emp", ascending=False)
        .head(TOP_N)
        .rename(columns={"tot_emp": "employment"})
        [["year", "state_name", "employment"]]
    )

    save_table(table_08, file_stem)

    save_barh_chart(
        table_08,
        label_col="state_name",
        value_col="employment",
        title=f"Top {TOP_N} States by Employment, {LATEST_YEAR}",
        xlabel="Employment",
        file_stem=file_stem,
        money=False
    )
else:
    print("SKIP 08 because no state-level 2025 rows were found.")

# ------------------------------------------------------------
# CHART 09
# Top industries by employment, 2025
# ------------------------------------------------------------

file_stem = "09_Top_industries_by_employment"

if not industry_2025_df.empty:
    if "naics_title" in industry_2025_df.columns:
        industry_2025_df["industry"] = industry_2025_df["naics_title"]
    else:
        industry_2025_df["industry"] = industry_2025_df.get("naics", "Unknown")

    key_col = "naics" if "naics" in industry_2025_df.columns else "industry"
    industry_best = dedupe_keep_highest_employment(industry_2025_df, [key_col])

    table_09 = (
        industry_best[industry_best["tot_emp"].notna()]
        .sort_values("tot_emp", ascending=False)
        .head(TOP_N)
        .rename(columns={"tot_emp": "employment"})
    )

    keep_cols = [c for c in ["year", "naics", "industry", "employment"] if c in table_09.columns]
    table_09 = table_09[keep_cols]

    save_table(table_09, file_stem)

    save_barh_chart(
        table_09,
        label_col="industry",
        value_col="employment",
        title=f"Top {TOP_N} Industries by Employment, {LATEST_YEAR}",
        xlabel="Employment",
        file_stem=file_stem,
        money=False
    )
else:
    print("SKIP 09 because no industry-level 2025 rows were found.")

# ------------------------------------------------------------
# CHART 10
# Top areas by employment, 2025
# ------------------------------------------------------------

file_stem = "10_Top_areas_by_employment"

if not area_2025_df.empty:
    if "area_title" in area_2025_df.columns:
        area_2025_df["area_name"] = area_2025_df["area_title"]
    else:
        area_2025_df["area_name"] = area_2025_df.get("area", "Unknown")

    key_col = "area" if "area" in area_2025_df.columns else "area_name"
    area_best = dedupe_keep_highest_employment(area_2025_df, [key_col])

    table_10 = (
        area_best[area_best["tot_emp"].notna()]
        .sort_values("tot_emp", ascending=False)
        .head(TOP_N)
        .rename(columns={"tot_emp": "employment"})
    )

    keep_cols = [c for c in ["year", "area", "area_name", "state_abbr", "employment"] if c in table_10.columns]
    table_10 = table_10[keep_cols]

    save_table(table_10, file_stem)

    save_barh_chart(
        table_10,
        label_col="area_name",
        value_col="employment",
        title=f"Top {TOP_N} Areas by Employment, {LATEST_YEAR}",
        xlabel="Employment",
        file_stem=file_stem,
        money=False
    )
else:
    print("SKIP 10 because no metro/nonmetro 2025 rows were found.")

# ------------------------------------------------------------
# CHART 11
# Hourly median wage by year
# ------------------------------------------------------------

file_stem = "11_Hourly_median_wage_by_year"

if "h_median" in overall_best.columns:
    table_11 = (
        overall_best[["year", "h_median"]]
        .dropna(subset=["year", "h_median"])
        .rename(columns={"h_median": "hourly_median_wage"})
        .sort_values("year")
    )

    table_11["year"] = table_11["year"].astype(int)

    save_table(table_11, file_stem)

    save_line_chart(
        table_11,
        x_col="year",
        y_col="hourly_median_wage",
        title="Hourly Median Wage by Year, 2010-2025",
        xlabel="Year",
        ylabel="Hourly median wage",
        file_stem=file_stem,
        money=True
    )
else:
    print("SKIP 11 because h_median column is missing.")

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print()
print("=" * 100)
print("DONE - ALL CHARTS CREATED")
print("=" * 100)
print("Charts saved to:", CHART_DIR)
print("Tables saved to:", TABLE_DIR)

print()
print("Expected chart files:")
for i in range(1, 12):
    print(f"{i:02d}_...")